<div style="background:linear-gradient(135deg,#00553A 0%,#00704A 55%,#00A86A 100%);padding:30px 34px;border-radius:14px;border-bottom:7px solid #F5C242;color:#FFFFFF">
<div style="color:#F5C242;font-weight:700;letter-spacing:3px;font-size:12px">ATELIER TECHNIQUE STG17 · JOUR 2 · 14 H 00 – 14 H 45 · LABORATOIRE</div>
<h1 style="color:#FFFFFF;margin:10px 0 6px 0;font-size:38px">Du bon à l’excellent</h1>
<h3 style="color:#E6F6EE;margin:0 0 14px 0;font-weight:400"><i>Optimiser les prompts en statistique officielle : un notebook pratique, pas à pas</i></h3>
<div style="color:#E6F6EE;font-size:13px">Enjeux émergents, pratiques émergentes · Innover dans la chaîne de valeur des données<br>Banque africaine de développement · Union africaine (STATAFRIC) · Institut national de la statistique du Rwanda</div>
</div>

## Pourquoi ce notebook ?

Ce matin, vous avez appris à **rédiger** un bon prompt. Ce notebook répond à la question suivante, celle que se pose tout institut national de statistique (INS) avant de mettre un LLM en production :

> **Comment prouver qu’un prompt fonctionne, l’améliorer de façon contrôlée, et en maîtriser le coût ?**

Nous travaillons sur un cas concret, connu de la plupart des INS : **la codification en CITP-08 des professions déclarées en texte libre** dans une enquête sur la population active (réponses en français et en anglais, saisies par des enquêteurs).

<div style="background:#E8F5EF;border-left:5px solid #00A86A;padding:12px 16px;border-radius:6px;margin:10px 0">
<b>🎯 À la fin de ce notebook, vous saurez :</b><br>
1. construire un <b>jeu d’évaluation</b> figé et représentatif ;<br>
2. <b>mesurer</b> la qualité d’un prompt avec la bonne métrique, et tenir un journal des itérations ;<br>
3. <b>maîtriser la variance</b> des réponses (température, vote majoritaire) ;<br>
4. estimer et réduire le <b>coût en jetons</b> (cache, traitement par lots, contexte court) ;<br>
5. choisir entre <b>prompt, RAG et affinage</b> selon l’échec observé.
</div>

### Plan

| Section | Contenu | Concept clé |
|---|---|---|
| **0** | Installation et configuration | Colab, Kaggle ou local ; mode réel ou simulation |
| **1** | Du bon à l’excellent | Un seul exemple ne prouve rien |
| **2** | Le jeu d’évaluation | Le test figé qui transforme les opinions en chiffres |
| **3** | Mesurer la qualité | Métriques, boucle d’optimisation, journal, fuite de données |
| **4** | Maîtriser la variance | Température, test de cohérence, vote majoritaire |
| **5** | Contexte, jetons et coûts | Jetons, coût, lots, cache, « perdu au milieu » |
| **6** | Prompt, RAG ou affinage ? | Mini-RAG, LLM juge, arbre de décision |
| **7** | 🧪 À vous de jouer | Optimiser le prompt vous-même |
| **8** | Synthèse et export | Liste de contrôle, fichiers pour le banc d’essai de 14 h 45 |

<div style="background:#FBF1D9;border-left:5px solid #D49A00;padding:12px 16px;border-radius:6px;margin:10px 0">
<b>⏱️ Comment utiliser ce notebook :</b> exécutez les cellules <b>dans l’ordre</b> (<i>Exécution → Tout exécuter</i> fonctionne aussi). Sans clé API, le notebook tourne en <b>mode simulation</b> : un simulateur pédagogique imite le comportement d’un LLM et réagit à vos modifications de prompt. Avec une clé Groq, tout s’exécute sur un vrai modèle.
</div>

<div style="background:#00704A;color:#FFFFFF;padding:14px 22px;border-radius:10px;border-left:10px solid #F5C242">
<span style="color:#F5C242;font-size:30px;font-weight:800">00</span>&nbsp;&nbsp;<span style="font-size:22px;font-weight:700">Installation et configuration</span><br>
<i style="color:#E6F6EE">Un seul notebook pour Colab, Kaggle et une installation locale.</i>
</div>

### 0.1 Installer les bibliothèques

La cellule suivante installe uniquement ce qui manque. Elle ne fait rien si tout est déjà présent (cas fréquent sur Colab et Kaggle).

In [ ]:
# ⚙️ Installation automatique des dépendances manquantes
import sys, os, subprocess, importlib

def _installer(paquet):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", paquet], check=False)

for module, paquet in [("openai", "openai>=1.40"), ("tiktoken", "tiktoken"), ("pandas", "pandas"),
                       ("matplotlib", "matplotlib"), ("sklearn", "scikit-learn")]:
    try:
        importlib.import_module(module)
    except ImportError:
        print(f"Installation de {paquet}…")
        _installer(paquet)

EN_COLAB = "google.colab" in sys.modules
EN_KAGGLE = os.path.exists("/kaggle") or "KAGGLE_KERNEL_RUN_TYPE" in os.environ
ENVIRONNEMENT = "Google Colab" if EN_COLAB else ("Kaggle" if EN_KAGGLE else "Local / autre")
print(f"✅ Dépendances prêtes · Environnement détecté : {ENVIRONNEMENT} · Python {sys.version.split()[0]}")

### 0.2 Choisir le fournisseur de modèle

| Option | Quand l’utiliser | Clé nécessaire |
|---|---|---|
| `"groq"` | Recommandé pour l’atelier (même moteur qu’à 14 h 45) | `GROQ_API_KEY` |
| `"openai"` | Tout fournisseur compatible OpenAI | `OPENAI_API_KEY` |
| `"ollama"` | Modèle ouvert **hébergé sur votre machine** : les données ne sortent pas | aucune |
| `"simulation"` | Sans connexion ni clé : simulateur pédagogique | aucune |
| `"auto"` | Groq si une clé est trouvée, sinon simulation | — |

**Où placer la clé ?**
- **Colab** : icône 🔑 *Secrets* à gauche → ajouter `GROQ_API_KEY` → activer l’accès pour ce notebook.
- **Kaggle** : *Add-ons → Secrets* → ajouter `GROQ_API_KEY`.
- **Local** : variable d’environnement `export GROQ_API_KEY=...` avant de lancer Jupyter.

<div style="background:#FBF1D9;border-left:5px solid #D49A00;padding:12px 16px;border-radius:6px;margin:10px 0">
<b>🔒 Confidentialité :</b> n’envoyez jamais de microdonnées identifiantes à une API externe sans avoir vérifié la base légale et les conditions d’utilisation des données. Les données de ce notebook sont <b>fictives</b>. Ne collez jamais une clé API en clair dans une cellule que vous partagez.
</div>

In [ ]:
# 🔧 CONFIGURATION : c'est la seule cellule à modifier
FOURNISSEUR = "auto"          # "auto" | "groq" | "openai" | "ollama" | "simulation"

MODELES = {
    "groq":   "openai/gpt-oss-20b",   # modèle de production Groq (vérifiez la liste à la cellule 0.3)
    "openai": "A_RENSEIGNER",         # nom exact du modèle chez votre fournisseur
    "ollama": "llama3.2",             # modèle téléchargé localement avec `ollama pull`
}
URL_OPENAI_COMPATIBLE = None      # ex. URL d'un autre fournisseur compatible OpenAI

# Prix ILLUSTRATIFS en dollars par million de jetons : remplacez-les par le tarif réel du fournisseur
PRIX_ENTREE_PAR_MILLION = 0.50
PRIX_SORTIE_PAR_MILLION = 1.50

DEMANDER_CLE_SI_ABSENTE = False   # True : saisie masquée de la clé si elle est introuvable
PAUSE_ENTRE_APPELS_S = 0.3        # petite pause pour respecter les limites de débit des offres gratuites

In [ ]:
# 🔑 Lecture sécurisée de la clé API (variables d'environnement, secrets Colab ou Kaggle)
import getpass

def lire_cle(nom):
    valeur = os.environ.get(nom)
    if valeur:
        return valeur
    if EN_COLAB:
        try:
            from google.colab import userdata
            return userdata.get(nom)
        except Exception:
            pass
    if EN_KAGGLE:
        try:
            from kaggle_secrets import UserSecretsClient
            return UserSecretsClient().get_secret(nom)
        except Exception:
            pass
    if DEMANDER_CLE_SI_ABSENTE:
        return getpass.getpass(f"{nom} (saisie masquée) : ") or None
    return None

CLE_GROQ = lire_cle("GROQ_API_KEY") if FOURNISSEUR in ("auto", "groq") else None
CLE_OPENAI = lire_cle("OPENAI_API_KEY") if FOURNISSEUR == "openai" else None

if FOURNISSEUR == "auto":
    FOURNISSEUR_ACTIF = "groq" if CLE_GROQ else "simulation"
else:
    FOURNISSEUR_ACTIF = FOURNISSEUR
if FOURNISSEUR_ACTIF == "groq" and not CLE_GROQ:
    print("⚠️ Aucune clé Groq trouvée : bascule en mode simulation.")
    FOURNISSEUR_ACTIF = "simulation"

MODELE_ACTIF = MODELES.get(FOURNISSEUR_ACTIF, "simulateur-pedagogique-v1")
MODE_SIMULATION = FOURNISSEUR_ACTIF == "simulation"
print(f"Fournisseur : {FOURNISSEUR_ACTIF} · Modèle : {MODELE_ACTIF}")
if MODE_SIMULATION:
    print("ℹ️ Mode simulation : les réponses viennent d'un simulateur pédagogique, pas d'un vrai LLM.")

### 0.3 Boîte à outils : style, comptage des jetons, client LLM et simulateur

Les cellules suivantes définissent les outils utilisés dans tout le notebook. **Vous n’avez pas besoin de les lire en détail** : exécutez-les. Retenez seulement ce que fait le client LLM :

- il envoie les messages au fournisseur choisi et **mesure** les jetons d’entrée et de sortie, la latence et le coût estimé ;
- il **met en cache** les réponses obtenues à température 0 (nous verrons pourquoi en section 5) ;
- il **consigne chaque appel** dans un journal, pour l’audit et le calcul des coûts ;
- il gère les **limites de débit** avec des nouvelles tentatives espacées.

In [ ]:
# 🎨 Style graphique (palette inspirée de la BAD) et fonctions d'affichage
import json, re, time, hashlib, random, math
from collections import Counter
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, HTML, Markdown

VERT, VERT_FONCE, FORET, OR, OCRE = "#00A86A", "#00704A", "#00553A", "#F5C242", "#D49A00"
SARCELLE, TERRE, BRIQUE, ENCRE, ARDOISE, SAUGE = "#0E7C86", "#C4621D", "#B83B2E", "#231F20", "#5E6964", "#D5DED9"
RAMPE = ["#9ED9C0", "#7BCBA9", "#57BD92", "#33AF7C", "#10A06A", "#008A5B", "#00664A"]

plt.rcParams.update({
    "figure.dpi": 110, "font.size": 10.5, "axes.edgecolor": SAUGE, "axes.labelcolor": ARDOISE,
    "xtick.color": ARDOISE, "ytick.color": ARDOISE, "axes.spines.top": False, "axes.spines.right": False,
    "axes.titleweight": "bold", "axes.titlesize": 12.5, "axes.titlecolor": ENCRE,
})
pd.set_option("display.max_colwidth", 80)

def encadre(texte, type_="note"):
    styles = {"note": ("#E8F5EF", VERT, "💡"), "attention": ("#FBF1D9", OCRE, "⚠️"),
              "retenir": ("#F4F7F5", VERT_FONCE, "📌"), "risque": ("#F6E3E0", BRIQUE, "⛔")}
    fond, bord, icone = styles[type_]
    display(HTML(f'<div style="background:{fond};border-left:5px solid {bord};padding:10px 14px;'
                 f'border-radius:6px;margin:8px 0;color:{ENCRE}">{icone} {texte}</div>'))

def valeur_sure(x):
    return "" if x is None else str(x)

In [ ]:
# 🔢 Comptage des jetons
try:
    import tiktoken
    _encodeur = tiktoken.get_encoding("o200k_base")
    def compter_jetons(texte):
        return len(_encodeur.encode(texte))
    METHODE_JETONS = "tiktoken o200k_base (proche du tokeniseur de gpt-oss)"
except Exception:
    def compter_jetons(texte):
        return max(1, round(len(texte) / 4))
    METHODE_JETONS = "approximation : 4 caractères ≈ 1 jeton (tiktoken indisponible)"

def cout_usd(jetons_entree, jetons_sortie):
    return jetons_entree / 1e6 * PRIX_ENTREE_PAR_MILLION + jetons_sortie / 1e6 * PRIX_SORTIE_PAR_MILLION

print("Méthode de comptage :", METHODE_JETONS)
print("Exemple :", compter_jetons("L'indice des prix à la consommation a augmenté de 2,4 % en juin."), "jetons")

In [ ]:
# 🤖 Simulateur pédagogique : imite un LLM et réagit au contenu du prompt
# Il ne "comprend" rien : il applique des règles qui reproduisent des comportements
# typiques des LLM (codes inventés sans liste, sur-interprétation des cas vagues,
# confusion entre catégories proches, sensibilité à la température, etc.).

class SimulateurLLM:
    MOTS_CLES = [
        ("2330", ["lycée", "secondary", "collège"]),
        ("2341", ["primary", "primaire", "instituteur", "institutri"]),
        ("2359", ["cours particuliers", "tutor"]),
        ("2211", ["docteur", "médecin", "doctor"]),
        ("2221", ["infirmi", "nurse"]),
        ("2512", ["software", "logiciel", "développeur", "developer"]),
        ("2411", ["comptable", "accountant"]),
        ("4132", ["saisie", "data entry"]),
        ("4110", ["administratif", "office clerk"]),
        ("5120", ["cook", "cuisinier", "cuisinière"]),
        ("5141", ["coiff", "hairdress"]),
        ("5311", ["nounou", "garde les enfants", "nanny", "child care"]),
        ("5414", ["gardien", "security", "vigile"]),
        ("5223", ["supermarket", "supermarché", "shop assistant"]),
        ("RUE_ALIM", ["beignet", "arachide", "cacahu", "grillé", "roasted"]),
        ("RUE", ["recharge", "airtime", "phone credit", "dans la rue", "on the street"]),
        ("5211", ["marché", "market"]),
        ("6121", ["chèvre", "vache", "bétail", "livestock", "cattle"]),
        ("6111", ["cultive", "maïs", "manioc", "farm", "champ"]),
        ("7112", ["maçon", "bricklayer", "mason"]),
        ("7115", ["menuisier", "charpentier", "carpenter"]),
        ("7231", ["mécanicien", "mechanic"]),
        ("7411", ["électricien", "electrician"]),
        ("7512", ["boulanger", "baker"]),
        ("7531", ["couturi", "tailleur", "tailor"]),
        ("8321", ["moto", "motorcycle"]),
        ("8322", ["taxi", "chauffeur", "driver"]),
        ("9112", ["cleaner", "nettoy", "entretien"]),
    ]
    VAGUES = {"ingénieur": "2142", "works": "9629", "aide ses parents": "6111",
              "business": "1420", "fonctionnaire": "4110", "travailleur": "9629"}
    MARQUEURS_MULTI = ["also", "aussi", "weekend", "week-end", "le soir"]

    @staticmethod
    def _h(texte):
        return int(hashlib.sha256(texte.encode()).hexdigest(), 16) % 100

    @staticmethod
    def _voisin(code):
        voisins = {"5211": "5221", "8321": "8322", "8322": "8321", "6111": "6121", "6121": "6111",
                   "2341": "2330", "2330": "2341", "7115": "7522", "7112": "7114", "5120": "5131",
                   "2221": "3221", "2211": "2240", "4132": "4131", "5414": "5419", "9112": "9111"}
        if code in voisins:
            return voisins[code]
        if code and code.isdigit():
            return code[:3] + str((int(code[3]) + 1) % 10)
        return "9629"

    def _mot_cle(self, t):
        for code, mots in self.MOTS_CLES:
            if any(m in t for m in mots):
                return code
        return None

    def _traits(self, prompt):
        p = prompt.lower()
        exemples = {m.group(1).strip().lower(): m.group(2) for m in re.finditer(r'-\s*"([^"]+)"\s*→\s*(\w+)', prompt)}
        return {
            "liste": "9520" in prompt and "8321" in prompt,
            "json": "json" in p,
            "schema": '"code_citp"' in prompt,
            "emploi_principal": "emploi principal" in p or "main job" in p,
            "non_codable": "NON_CODABLE" in prompt and ("trop vague" in p or "imprécis" in p),
            "supermarche": bool(re.search(r"supermarch|supermarket", re.sub(r'-\s*"[^"]+"\s*→\s*\w+', "", p))),
            "exemples": exemples,
        }

    def _coder(self, texte, tr, temperature, rng, sans_exemples=False):
        t = texte.strip().lower()
        exemples = {} if sans_exemples else tr["exemples"]
        codes_ex = set(exemples.values())
        if t in exemples:                                   # le cas est dans le prompt : "fuite"
            return exemples[t], False
        if t in self.VAGUES:
            return ("NON_CODABLE" if tr["non_codable"] else self.VAGUES[t]), False
        multi = "," in t and any(m in t for m in self.MARQUEURS_MULTI)
        if multi:
            principal, secondaire = t.split(",", 1)
            c1, c2 = self._mot_cle(principal), self._mot_cle(secondaire)
            brut = c1 if (tr["emploi_principal"] or c2 is None) else c2
        else:
            brut = self._mot_cle(t)
        code = brut
        if brut == "RUE_ALIM":
            code = "5212" if "5212" in codes_ex else "5211"
        elif brut == "RUE":
            code = "9520" if "9520" in codes_ex else "5211"
        elif brut == "5223" and not tr["supermarche"]:
            code = "5211"
        if code is None:
            code = "9629"
        h = self._h(t)
        if not tr["liste"] and h < 45:
            code = code[:3] + "0"                            # code inventé ou mauvaise granularité
        elif h < 6:
            code = self._voisin(code)                        # cas réellement difficile
        if temperature > 0:
            p = temperature * (0.35 if (multi or brut in ("RUE", "RUE_ALIM", "5223")) else 0.12)
            if rng.random() < p:
                code = rng.choice([self._voisin(code), "5211", "9629"])
        return code, multi

    def _codage(self, messages, prompt, temperature, rng):
        systeme = messages[0]["content"] if messages[0]["role"] == "system" else ""
        tr = self._traits(systeme)   # le simulateur ne "lit" que les consignes, pas le cas à coder
        utilisateur = messages[-1]["content"]
        lots = re.findall(r"^\[(\w+)\]\s*(.+)$", utilisateur, flags=re.M)
        if lots:
            resultats = []
            for i, (ident, texte) in enumerate(lots):
                code, _ = self._coder(texte, tr, temperature, rng, sans_exemples=(i >= 8))
                resultats.append({"id": ident, "code_citp": code})
            return json.dumps({"resultats": resultats}, ensure_ascii=False)
        m = re.search(r"Description\s*:\s*(.+)", utilisateur)
        texte = m.group(1) if m else re.split(r":\s*", utilisateur)[-1]
        code, multi = self._coder(texte, tr, temperature, rng)
        confiance = "faible" if code == "NON_CODABLE" else ("moyenne" if multi else "haute")
        if tr["json"] and tr["schema"]:
            return json.dumps({"code_citp": code, "confiance": confiance}, ensure_ascii=False)
        if tr["json"]:
            return json.dumps({"code": code}, ensure_ascii=False)
        if not tr["liste"]:
            return (f"Cette profession correspond probablement au code CITP {code}. "
                    "Il faudrait toutefois vérifier dans la nomenclature officielle, car plusieurs "
                    "groupes peuvent convenir selon le contexte de l'emploi.")
        return f"D'après la liste fournie, le code CITP-08 le plus approprié est {code}."

    def _aiguille(self, prompt, question):
        cible = re.search(r"mars s'établit à ([\d,]+) %", prompt)
        leurre = re.search(r"février s'établit à ([\d,]+) %", prompt)
        if not cible:
            return "Je ne trouve pas cette information dans le bulletin."
        position = cible.start() / max(1, len(prompt))
        long_contexte = compter_jetons(prompt) > 1500
        if long_contexte and 0.25 < position < 0.75 and self._h(cible.group(1)) % 3 != 0 and leurre:
            return f"{leurre.group(1)} %"
        return f"{cible.group(1)} %"

    def _juge(self, utilisateur):
        r1 = re.search(r"RÉSUMÉ 1 :\n(.+?)\n\nRÉSUMÉ 2", utilisateur, flags=re.S).group(1)
        r2 = re.search(r"RÉSUMÉ 2 :\n(.+?)$", utilisateur, flags=re.S).group(1)
        def score(r, pos):
            return (2.0 if "12 480" in r else 0) + len(r) / 150 + (1.1 if pos == 1 else 0)
        meilleur = 1 if score(r1, 1) >= score(r2, 2) else 2
        return json.dumps({"meilleur": meilleur, "justification": "Résumé plus complet et mieux présenté."},
                          ensure_ascii=False)

    def _rag(self, prompt):
        if "12 480" in prompt:
            return "L'échantillon de l'EPA 2025 comprend 12 480 ménages [P3]."
        return "L'enquête EPA 2025 de Fictivia porte sur un échantillon d'environ 10 000 ménages."

    def repondre(self, messages, temperature=0.0, json_mode=False):
        prompt = "\n".join(m["content"] for m in messages)
        systeme = messages[0]["content"] if messages[0]["role"] == "system" else ""
        rng = random.Random() if temperature > 0 else random.Random(0)
        if "Fictivia" in prompt:
            texte = self._rag(prompt)
        elif "BULLETIN" in prompt:
            texte = self._aiguille(prompt, messages[-1]["content"])
        elif "ÉVALUATEUR" in systeme:
            texte = self._juge(messages[-1]["content"])
        elif "CITP" in prompt or "profession" in prompt.lower():
            texte = self._codage(messages, prompt, temperature, rng)
        else:
            texte = "Réponse simulée."
        je, js = compter_jetons(prompt) + 4 * len(messages), compter_jetons(texte)
        latence = 0.12 + 0.004 * js + 0.00004 * je
        return texte, je, js, 0, latence

SIMULATEUR = SimulateurLLM()
print("✅ Simulateur prêt.")

In [ ]:
# 🔌 Client LLM unifié : mesure, cache, journal des appels, nouvelles tentatives
class ClientLLM:
    URLS = {"groq": "https://api.groq.com/openai/v1", "ollama": "http://localhost:11434/v1"}

    def __init__(self, fournisseur, modele, cle=None, base_url=None):
        self.fournisseur, self.modele = fournisseur, modele
        self.cache, self.appels = {}, []
        self._client = None
        if fournisseur != "simulation":
            from openai import OpenAI
            url = base_url or self.URLS.get(fournisseur)
            self._client = OpenAI(api_key=cle or "cle-locale", base_url=url)

    def _cle_cache(self, messages, temperature, json_mode, max_tokens):
        brut = json.dumps([self.fournisseur, self.modele, messages, temperature, json_mode, max_tokens],
                          ensure_ascii=False, sort_keys=True)
        return hashlib.sha256(brut.encode()).hexdigest()

    def _appel_api(self, messages, temperature, max_tokens, json_mode):
        args = dict(model=self.modele, messages=messages, temperature=temperature, max_tokens=max_tokens)
        if json_mode:
            args["response_format"] = {"type": "json_object"}
        if self.fournisseur == "groq" and "gpt-oss" in self.modele:
            args["extra_body"] = {"reasoning_effort": "low"}
        for tentative in range(6):
            try:
                reponse = self._client.chat.completions.create(**args)
                break
            except Exception as erreur:
                message = str(erreur).lower()
                if "reasoning" in message and "extra_body" in args:
                    args.pop("extra_body"); continue
                if "response_format" in message and "response_format" in args:
                    args.pop("response_format"); continue
                if tentative < 5 and any(k in message for k in ["rate", "429", "timeout", "503", "overloaded", "connection"]):
                    attente = 2 ** tentative * 2
                    print(f"   ⏳ Limite de débit ou erreur temporaire : nouvelle tentative dans {attente} s")
                    time.sleep(attente); continue
                raise
        texte = reponse.choices[0].message.content or ""
        usage = reponse.usage
        details = getattr(usage, "prompt_tokens_details", None)
        en_cache = (getattr(details, "cached_tokens", 0) or 0) if details else 0
        if PAUSE_ENTRE_APPELS_S:
            time.sleep(PAUSE_ENTRE_APPELS_S)
        return texte, usage.prompt_tokens or 0, usage.completion_tokens or 0, en_cache

    def chat(self, messages, temperature=0.0, max_tokens=1024, json_mode=False, utiliser_cache=True, etiquette=""):
        cle = self._cle_cache(messages, temperature, json_mode, max_tokens)
        if utiliser_cache and temperature == 0 and cle in self.cache:
            r = dict(self.cache[cle], depuis_cache=True, latence_s=0.0, jetons_entree=0, jetons_sortie=0,
                     jetons_en_cache_fournisseur=0, cout_usd=0.0)
            self.appels.append(dict(r, etiquette=etiquette, texte=None))
            return r
        debut = time.perf_counter()
        if self.fournisseur == "simulation":
            texte, je, js, jc, latence = SIMULATEUR.repondre(messages, temperature, json_mode)
        else:
            texte, je, js, jc = self._appel_api(messages, temperature, max_tokens, json_mode)
            latence = time.perf_counter() - debut
        r = dict(texte=texte, jetons_entree=je, jetons_sortie=js, jetons_en_cache_fournisseur=jc,
                 latence_s=round(latence, 3), cout_usd=cout_usd(je, js), depuis_cache=False)
        if temperature == 0:
            self.cache[cle] = r
        self.appels.append(dict(r, etiquette=etiquette, texte=None))
        return r

    def bilan(self, etiquette=None):
        df = pd.DataFrame(self.appels)
        if df.empty:
            return df
        if etiquette:
            df = df[df["etiquette"] == etiquette]
        return (df.groupby("etiquette")
                  .agg(appels=("etiquette", "size"), depuis_cache=("depuis_cache", "sum"),
                       jetons_entree=("jetons_entree", "sum"), jetons_sortie=("jetons_sortie", "sum"),
                       cout_usd=("cout_usd", "sum"), latence_moy_s=("latence_s", "mean"))
                  .round(4))

LLM = ClientLLM(FOURNISSEUR_ACTIF, MODELE_ACTIF,
                cle=CLE_GROQ if FOURNISSEUR_ACTIF == "groq" else CLE_OPENAI,
                base_url=URL_OPENAI_COMPATIBLE if FOURNISSEUR_ACTIF == "openai" else None)

test = LLM.chat([{"role": "user", "content": "Réponds seulement : OK."}], etiquette="test")
print("✅ Client prêt · Réponse de test :", test["texte"][:80], "·", test["jetons_entree"], "jetons en entrée")

In [ ]:
# 📋 (Optionnel, mode réel) Lister les modèles actifs chez le fournisseur
if not MODE_SIMULATION:
    try:
        modeles = sorted(m.id for m in LLM._client.models.list().data)
        print(f"{len(modeles)} modèles disponibles :")
        print(" · ".join(modeles))
    except Exception as e:
        print("Impossible de lister les modèles :", e)
else:
    print("Mode simulation : aucune liste de modèles à afficher.")

<div style="background:#00A86A;color:#FFFFFF;padding:14px 22px;border-radius:10px;border-left:10px solid #F5C242">
<span style="color:#F5C242;font-size:30px;font-weight:800">01</span>&nbsp;&nbsp;<span style="font-size:22px;font-weight:700">Du bon à l’excellent</span><br>
<i style="color:#E6F6EE">Pourquoi un prompt efficace hier échoue-t-il sur le bulletin du jour ?</i>
</div>

### 1.1 L’expérience que tout le monde fait

On écrit un prompt, on l’essaie sur **un** exemple, la réponse semble correcte… et on conclut que « ça marche ». Faisons exactement cela.

In [ ]:
prompt_naif = "Quel est le code CITP de cette profession : {texte}"

exemple_facile = "Vendeuse de tomates au marché central"
r = LLM.chat([{"role": "user", "content": prompt_naif.format(texte=exemple_facile)}], etiquette="section1")
print("Entrée  :", exemple_facile)
print("Réponse :", r["texte"])
print("Attendu : 5211 (vendeurs sur les marchés et éventaires)")

La réponse **semble** plausible. Mais regardez de près : est-elle exploitable automatiquement ? Le code cité existe-t-il vraiment dans la CITP-08 ? Essayons maintenant quelques descriptions réelles, telles que les enquêteurs les saisissent.

In [ ]:
quelques_cas = [
    ("Vend des cartes de recharge dans la rue", "9520"),
    ("Primary teacher, sells airtime at weekends", "2341"),
    ("Ingénieur", "NON_CODABLE"),
    ("Moto-taxi driver, owns the motorcycle", "8321"),
]
for texte, attendu in quelques_cas:
    r = LLM.chat([{"role": "user", "content": prompt_naif.format(texte=texte)}], etiquette="section1")
    print(f"• {texte}\n  attendu : {attendu}\n  réponse : {r['texte'][:160]}\n")

<div style="background:#F4F7F5;border-left:5px solid #00704A;padding:12px 16px;border-radius:6px">
<b>📌 Ce que l’on observe</b> (en simulation comme avec la plupart des vrais modèles) :
<ul>
<li>des réponses <b>en prose</b>, impossibles à intégrer automatiquement dans une chaîne de production ;</li>
<li>des codes <b>inventés</b> ou au mauvais niveau de détail, faute de liste de référence ;</li>
<li>un code donné pour « Ingénieur » alors que l’information est <b>insuffisante</b> ;</li>
<li>une confusion quand la personne a <b>deux activités</b>.</li>
</ul>
Un seul exemple ne prouve rien. Pour la statistique officielle, les <b>Principes fondamentaux de l’ONU</b> (résolution 68/261, 2014) demandent des méthodes choisies selon des critères professionnels et scientifiques (principe 2). Une étape LLM en production doit satisfaire la même exigence : <b>un taux d’erreur mesuré</b>.
</div>

| | Un « bon » prompt | Un prompt « excellent » |
|---|---|---|
| **Testé sur** | l’exemple essayé par hasard | un jeu figé de 30 à 50 cas représentatifs |
| **Jugé par** | une impression favorable | des critères écrits et un score chiffré |
| **Taux d’erreur** | inconnu | mesuré, avec des types d’échec identifiés |
| **Reproductibilité** | réponses qui dérivent | modèle figé, prompt versionné |
| **Coût** | découvert sur la facture | estimé à l’avance pour 1 000 documents |

<div style="background:#00704A;color:#FFFFFF;padding:14px 22px;border-radius:10px;border-left:10px solid #F5C242">
<span style="color:#F5C242;font-size:30px;font-weight:800">02</span>&nbsp;&nbsp;<span style="font-size:22px;font-weight:700">Le jeu d’évaluation</span><br>
<i style="color:#E6F6EE">Comment saurez-vous que la version 2 est meilleure que la version 1 ?</i>
</div>

### 2.1 Ce qu’est un jeu d’évaluation

Un **jeu d’évaluation** est un ensemble **figé** de cas, chacun avec une **réponse de référence** validée par des experts. C’est l’équivalent, pour un prompt, d’un échantillon de contrôle en codification manuelle.

Nous utilisons trois ensembles **distincts** :

| Ensemble | Rôle | Taille ici |
|---|---|---|
| 🟢 **Réservoir d’exemples** | cas que l’on a le droit de copier dans le prompt | 6 |
| 🔵 **Jeu d’évaluation** | cas sur lesquels on mesure chaque version | 30 |
| 🟠 **Jeu réservé** | cas consultés rarement, pour confirmer les gains réels | 8 |

Et trois **types de cas** :
- **typiques** (~60 %) : le parcours courant, dans toutes les langues ;
- **limites** (~25 %) : là où les règles de décision sont mises à l’épreuve (deux emplois, catégories proches) ;
- **négatifs** (~15 %) : le modèle doit répondre `NON_CODABLE` au lieu d’inventer un code.

In [ ]:
# 📚 Liste de référence (libellés abrégés de groupes de base CITP-08, OIT 2012)
LISTE_CITP = {
    "2211": "Médecins généralistes", "2221": "Personnel infirmier (niveau professionnel)",
    "2330": "Professeurs de l'enseignement secondaire", "2341": "Instituteurs de l'enseignement primaire",
    "2411": "Comptables", "2512": "Concepteurs de logiciels", "4110": "Employés de bureau, fonctions générales",
    "4132": "Opérateurs de saisie", "5120": "Cuisiniers", "5141": "Coiffeurs",
    "5211": "Vendeurs sur les marchés et éventaires", "5212": "Vendeurs ambulants de produits alimentaires",
    "5223": "Vendeurs et assistants de vente en magasin", "5311": "Gardes d'enfants",
    "5414": "Agents de sécurité", "6111": "Agriculteurs, cultures de plein champ",
    "6121": "Éleveurs de bétail et producteurs laitiers", "7112": "Maçons",
    "7115": "Charpentiers et menuisiers du bâtiment", "7231": "Mécaniciens de véhicules à moteur",
    "7411": "Électriciens du bâtiment", "7512": "Boulangers, pâtissiers et confiseurs",
    "7531": "Tailleurs, couturiers", "8321": "Conducteurs de motocycles",
    "8322": "Conducteurs d'automobiles, de taxis et de camionnettes",
    "9112": "Agents d'entretien dans les bureaux et hôtels", "9520": "Vendeurs ambulants (hors alimentation)",
}
TEXTE_LISTE = "\n".join(f"- {c} : {l}" for c, l in LISTE_CITP.items())

# 🟢 Réservoir d'exemples (utilisables dans le prompt)
RESERVOIR_EXEMPLES = [
    ("Vendeur de poisson au marché", "5211"),
    ("Conduit une moto pour transporter des passagers", "8321"),
    ("Teacher at primary school, also farms on weekends", "2341"),
    ("Vend des arachides grillées dans la rue", "5212"),
    ("Fonctionnaire", "NON_CODABLE"),
    ("Sells phone credit on the street", "9520"),
]

# 🔵 Jeu d'évaluation : 30 cas figés
_cas = [
    ("E01", "Vendeuse de tomates au marché central", "fr", "5211", "typique"),
    ("E02", "Moto-taxi driver, owns the motorcycle", "en", "8321", "typique"),
    ("E03", "Agent de saisie à l'institut de statistique", "fr", "4132", "typique"),
    ("E04", "Cultive du maïs et du manioc sur son champ", "fr", "6111", "typique"),
    ("E05", "Taxi driver in the capital", "en", "8322", "typique"),
    ("E06", "Couturière dans un atelier de quartier", "fr", "7531", "typique"),
    ("E07", "Maçon sur les chantiers de construction", "fr", "7112", "typique"),
    ("E08", "Hairdresser in a beauty salon", "en", "5141", "typique"),
    ("E09", "Infirmière diplômée à l'hôpital régional", "fr", "2221", "typique"),
    ("E10", "Software developer for a mobile banking company", "en", "2512", "typique"),
    ("E11", "Gardien de nuit dans une banque", "fr", "5414", "typique"),
    ("E12", "Mécanicien automobile dans un garage", "fr", "7231", "typique"),
    ("E13", "Boulanger, fabrique le pain chaque matin", "fr", "7512", "typique"),
    ("E14", "Cook in a hotel restaurant", "en", "5120", "typique"),
    ("E15", "Comptable dans une PME", "fr", "2411", "typique"),
    ("E16", "Élève des chèvres et des vaches", "fr", "6121", "typique"),
    ("E17", "Électricien, installe le câblage des maisons", "fr", "7411", "typique"),
    ("E18", "Cleaner in government offices", "en", "9112", "typique"),
    ("L01", "Primary teacher, sells airtime at weekends", "en", "2341", "limite"),
    ("L02", "Vend des beignets au bord de la route", "fr", "5212", "limite"),
    ("L03", "Vend des cartes de recharge dans la rue", "fr", "9520", "limite"),
    ("L04", "Enseignante au lycée, donne des cours particuliers le soir", "fr", "2330", "limite"),
    ("L05", "Nounou, garde les enfants des voisins", "fr", "5311", "limite"),
    ("L06", "Shop assistant in a supermarket", "en", "5223", "limite"),
    ("L07", "Agent administratif à la mairie", "fr", "4110", "limite"),
    ("L08", "Docteur au centre de santé", "fr", "2211", "limite"),
    ("N01", "Ingénieur", "fr", "NON_CODABLE", "négatif"),
    ("N02", "works", "en", "NON_CODABLE", "négatif"),
    ("N03", "Aide ses parents", "fr", "NON_CODABLE", "négatif"),
    ("N04", "Business", "en", "NON_CODABLE", "négatif"),
]
JEU_EVAL = pd.DataFrame(_cas, columns=["id", "texte", "langue", "code_ref", "type_cas"])

# 🟠 Jeu réservé : on ne le regarde qu'occasionnellement
JEU_RESERVE = pd.DataFrame([
    ("H01", "Vendeur de pagnes au grand marché", "fr", "5211", "typique"),
    ("H02", "Conducteur de taxi-moto", "fr", "8321", "typique"),
    ("H03", "Institutrice, vend aussi des beignets le week-end", "fr", "2341", "limite"),
    ("H04", "Vend des cacahuètes grillées au bord de la route", "fr", "5212", "limite"),
    ("H05", "Nurse at a private clinic", "en", "2221", "typique"),
    ("H06", "Travailleur", "fr", "NON_CODABLE", "négatif"),
    ("H07", "Vendeuse dans un supermarché", "fr", "5223", "limite"),
    ("H08", "Mechanic repairing cars and trucks", "en", "7231", "typique"),
], columns=["id", "texte", "langue", "code_ref", "type_cas"])

# 🔒 "Geler" le jeu : son empreinte change si un seul caractère est modifié
def empreinte(df):
    return hashlib.sha256(df.to_csv(index=False).encode()).hexdigest()[:12]

EMPREINTE_EVAL = empreinte(JEU_EVAL)
print(f"Jeu d'évaluation : {len(JEU_EVAL)} cas · empreinte {EMPREINTE_EVAL}")
print(f"Jeu réservé      : {len(JEU_RESERVE)} cas · empreinte {empreinte(JEU_RESERVE)}")
display(JEU_EVAL.head(8))

In [ ]:
# 📊 Le jeu est-il représentatif ? Répartition par type de cas et par langue
fig, axes = plt.subplots(1, 2, figsize=(10, 3.2))
types = JEU_EVAL["type_cas"].value_counts().reindex(["typique", "limite", "négatif"])
axes[0].barh(types.index, types.values, color=[VERT, OCRE, BRIQUE])
for i, v in enumerate(types.values):
    axes[0].text(v + 0.3, i, f"{v} ({v/len(JEU_EVAL):.0%})", va="center", color=ENCRE)
axes[0].set_title("Types de cas"); axes[0].invert_yaxis(); axes[0].set_xlim(0, 23)
langues = JEU_EVAL["langue"].value_counts()
axes[1].bar(langues.index.str.upper(), langues.values, color=[VERT_FONCE, SARCELLE])
for i, v in enumerate(langues.values):
    axes[1].text(i, v + 0.3, str(v), ha="center", color=ENCRE)
axes[1].set_title("Langues des réponses"); axes[1].set_ylim(0, 23)
plt.tight_layout(); plt.show()

<div style="background:#F4F7F5;border-left:5px solid #00704A;padding:12px 16px;border-radius:6px">
<b>📌 Cinq règles pour un jeu d’évaluation fiable</b>
<ol>
<li><b>Représentatif</b> : chaque langue, région, source et format rencontrés en production, y compris les fautes de frappe.</li>
<li><b>Annoté par des experts</b> : double codification, désaccords résolus avant la notation.</li>
<li><b>Figé et versionné</b> : aucune modification une fois la notation lancée (d’où l’empreinte ci-dessus), stockage dans Git.</li>
<li><b>Jamais dans le prompt</b> : les exemples du prompt viennent d’un autre réservoir (nous verrons en 3.5 ce qui se passe sinon).</li>
<li><b>Nourri par les échecs</b> : chaque erreur rencontrée en production devient un nouveau cas.</li>
</ol>
<i>Règle empirique : commencer avec 30 à 50 cas.</i>
</div>

<div style="background:#0E7C86;color:#FFFFFF;padding:14px 22px;border-radius:10px;border-left:10px solid #F5C242">
<span style="color:#F5C242;font-size:30px;font-weight:800">03</span>&nbsp;&nbsp;<span style="font-size:22px;font-weight:700">Mesurer la qualité</span><br>
<i style="color:#E6F6EE">Mesurer, plutôt que ressentir.</i>
</div>

### 3.1 Choisir la métrique adaptée à la tâche

| Type de tâche | Exemple dans un INS | Métrique | Notation |
|---|---|---|---|
| Extraction | chiffres de l’IPC tirés d’un bulletin PDF | correspondance exacte par champ | automatique |
| **Codification** ⬅️ *notre cas* | CITP, CITI, COICOP | **exactitude**, précision / rappel par classe | automatique + matrice de confusion |
| Sortie structurée | JSON pour un tableau de bord | **taux de JSON valide** | automatique |
| Questions-réponses (RAG) | notes méthodologiques | fidélité, exactitude des citations | grille, humain ou juge calibré |
| Rédaction | communiqué de presse | grille 1 à 5 | relecteurs, juge calibré |

Pour notre tâche, nous mesurons **deux exactitudes** :
- **stricte** : la réponse doit être un JSON valide contenant le bon code. C’est ce qui compte en production, car une réponse illisible par la machine est un échec ;
- **souple** : on cherche un code n’importe où dans le texte. Elle montre ce que le modèle « sait », indépendamment du format.

In [ ]:
# 🧮 Moteur d'évaluation
JOURNAL = []   # journal des itérations : une ligne par version testée

def extraire_code(texte):
    """Renvoie (code_strict, code_souple, json_valide)."""
    brut = re.sub(r"^```(?:json)?|```$", "", (texte or "").strip(), flags=re.M).strip()
    code_strict, json_valide = None, False
    try:
        objet = json.loads(brut)
        if isinstance(objet, dict) and objet.get("code_citp"):
            code_strict, json_valide = str(objet["code_citp"]).strip().upper(), True
    except Exception:
        pass
    m = re.search(r"\b(\d{4}|NON_CODABLE)\b", texte or "")
    return code_strict, (m.group(1) if m else None), json_valide

def evaluer(version, construire_messages, jeu=None, json_mode=False, temperature=0.0,
            changement="", consigner=True, afficher=True, utiliser_cache=True):
    jeu = JEU_EVAL if jeu is None else jeu
    lignes = []
    for cas in jeu.itertuples():
        r = LLM.chat(construire_messages(cas.texte), temperature=temperature, json_mode=json_mode,
                     etiquette=version, utiliser_cache=utiliser_cache)
        strict, souple, valide = extraire_code(r["texte"])
        lignes.append(dict(id=cas.id, texte=cas.texte, type_cas=cas.type_cas, code_ref=cas.code_ref,
                           code_obtenu=strict or souple, json_valide=valide,
                           correct_strict=(strict == cas.code_ref), correct_souple=(souple == cas.code_ref),
                           jetons_entree=r["jetons_entree"], jetons_sortie=r["jetons_sortie"],
                           cout_usd=r["cout_usd"], reponse=r["texte"]))
    df = pd.DataFrame(lignes)
    resume = dict(version=version, changement=changement, jeu="réservé" if jeu is JEU_RESERVE else "évaluation",
                  exactitude_stricte=df["correct_strict"].mean(), exactitude_souple=df["correct_souple"].mean(),
                  json_valide=df["json_valide"].mean(),
                  jetons_par_cas=df["jetons_entree"].sum() / len(df) if df["jetons_entree"].sum() else None,
                  cout_pour_1000_usd=df["cout_usd"].sum() / len(df) * 1000 if df["cout_usd"].sum() else None,
                  decision="")
    for t in ["typique", "limite", "négatif"]:
        sous = df[df["type_cas"] == t]
        resume[f"exact_{t}"] = sous["correct_strict"].mean() if len(sous) else None
    if consigner:
        JOURNAL[:] = [j for j in JOURNAL if not (j["version"] == version and j["jeu"] == resume["jeu"])]
        JOURNAL.append(resume)
    if afficher:
        carte_score(resume)
    return df, resume

def carte_score(res):
    def tuile(titre, valeur, couleur):
        return (f'<div style="flex:1;background:#F4F7F5;border:1px solid {SAUGE};border-radius:10px;padding:10px 14px;margin:4px">'
                f'<div style="font-size:11px;letter-spacing:1px;color:{ARDOISE};font-weight:700">{titre}</div>'
                f'<div style="font-size:26px;font-weight:800;color:{couleur}">{valeur}</div></div>')
    pct = lambda x: "—" if x is None or (isinstance(x, float) and math.isnan(x)) else f"{x:.0%}"
    cout = "—" if not res["cout_pour_1000_usd"] else f"{res['cout_pour_1000_usd']:.3f} $"
    html = (f'<div style="font-weight:800;color:{VERT_FONCE};margin-top:6px">VERSION {res["version"]} · jeu {res["jeu"]}'
            f'<span style="font-weight:400;color:{ARDOISE}"> · {res["changement"]}</span></div>'
            f'<div style="display:flex;flex-wrap:wrap">'
            + tuile("EXACTITUDE STRICTE", pct(res["exactitude_stricte"]), VERT_FONCE)
            + tuile("EXACTITUDE SOUPLE", pct(res["exactitude_souple"]), SARCELLE)
            + tuile("JSON VALIDE", pct(res["json_valide"]), OCRE)
            + tuile("TYPIQUES · LIMITES · NÉGATIFS",
                    f'{pct(res["exact_typique"])} · {pct(res["exact_limite"])} · {pct(res["exact_négatif"])}', ENCRE)
            + tuile("COÛT / 1 000 CAS", cout, TERRE) + "</div>")
    display(HTML(html))

def decider(version, decision):
    for j in JOURNAL:
        if j["version"] == version:
            j["decision"] = decision

def afficher_erreurs(df, n=10):
    err = df[~df["correct_strict"]][["id", "type_cas", "texte", "code_ref", "code_obtenu", "json_valide"]]
    if err.empty:
        encadre("Aucune erreur sur ce jeu.", "note")
    else:
        display(err.head(n))

print("✅ Moteur d'évaluation prêt.")

### 3.2 La boucle d’optimisation

<div style="display:flex;flex-wrap:wrap;gap:6px;margin:8px 0">
<div style="flex:1;min-width:120px;background:#F4F7F5;border:1px solid #D5DED9;border-radius:8px;padding:8px"><b style="color:#00704A">1 · Définir le succès</b><br><small>critères et score cible</small></div>
<div style="flex:1;min-width:120px;background:#F4F7F5;border:1px solid #D5DED9;border-radius:8px;padding:8px"><b style="color:#00704A">2 · Constituer le jeu</b><br><small>fait en section 2</small></div>
<div style="flex:1;min-width:120px;background:#F4F7F5;border:1px solid #D5DED9;border-radius:8px;padding:8px"><b style="color:#00704A">3 · Mesurer la base</b><br><small>score de v0</small></div>
<div style="flex:1;min-width:120px;background:#F4F7F5;border:1px solid #D5DED9;border-radius:8px;padding:8px"><b style="color:#00704A">4 · Diagnostiquer</b><br><small>lire les échecs</small></div>
<div style="flex:1;min-width:120px;background:#E8F5EF;border:2px solid #D49A00;border-radius:8px;padding:8px"><b style="color:#D49A00">5 · Changer UNE chose</b><br><small>règle, exemple ou paramètre</small></div>
<div style="flex:1;min-width:120px;background:#F4F7F5;border:1px solid #D5DED9;border-radius:8px;padding:8px"><b style="color:#00704A">6 · Re-tester, consigner</b><br><small>journal des itérations</small></div>
</div>

**Étape 1 : définir le succès avant d’écrire le moindre prompt.**

In [ ]:
CIBLE_EXACTITUDE = 0.90   # exactitude stricte minimale sur le jeu d'évaluation
CIBLE_JSON = 1.00         # toutes les réponses doivent être exploitables par la machine
print(f"🎯 Objectif : exactitude stricte ≥ {CIBLE_EXACTITUDE:.0%} et JSON valide = {CIBLE_JSON:.0%}")

### 3.3 Version 0 : la base

Le prompt naïf de la section 1. Nous le mesurons sur les 30 cas : c’est notre **point de référence**.

In [ ]:
def messages_v0(texte):
    return [{"role": "user", "content": f"Quel est le code CITP de cette profession : {texte}"}]

df_v0, res_v0 = evaluer("v0", messages_v0, changement="prompt naïf (base)")
decider("v0", "base")
afficher_erreurs(df_v0, 6)

**Étape 4 : diagnostiquer.** Trois causes ressortent : (a) des réponses en prose, donc JSON valide = 0 % et exactitude stricte nulle ; (b) des codes inventés, faute de liste ; (c) aucune consigne pour les cas vagues ou multiples.

**Étape 5 : ne changer qu’une chose.** Commençons par la cause la plus structurante : donner au modèle un **rôle**, un **objectif**, le **contexte** et la **liste de référence**.

### 3.4 Versions 1 à 4 : un changement à la fois

In [ ]:
SYSTEME_V1 = f"""Tu es codificateur statistique à l'Institut national de la statistique.
OBJECTIF : attribuer à chaque description d'emploi le code CITP-08 à quatre chiffres (groupe de base) le plus approprié.
CONTEXTE : les descriptions proviennent de l'enquête sur la population active. Elles sont rédigées en français ou en anglais par des enquêteurs, souvent de façon abrégée.
Utilise uniquement les codes de la liste suivante :
{TEXTE_LISTE}"""

def messages_v1(texte):
    return [{"role": "system", "content": SYSTEME_V1},
            {"role": "user", "content": f"Description : {texte}"}]

df_v1, res_v1 = evaluer("v1", messages_v1, changement="+ rôle, objectif, contexte, liste de codes")

L’exactitude **souple** progresse (moins de codes inventés), mais l’exactitude **stricte** reste nulle : la réponse n’est toujours pas exploitable par une machine. Changement suivant : **imposer le format**. Nous activons aussi le mode JSON de l’API (`json_mode=True`), un levier côté paramètres.

In [ ]:
FORMAT_JSON = """
FORMAT DE SORTIE
Réponds uniquement avec un objet JSON, sans aucun texte autour :
{"code_citp": "<code à 4 chiffres>", "confiance": "haute|moyenne|faible"}"""

SYSTEME_V2 = SYSTEME_V1 + "\n" + FORMAT_JSON

def messages_v2(texte):
    return [{"role": "system", "content": SYSTEME_V2},
            {"role": "user", "content": f"Description : {texte}"}]

df_v2, res_v2 = evaluer("v2", messages_v2, json_mode=True, changement="+ format JSON imposé")
afficher_erreurs(df_v2, 10)

Le format est réglé. Les erreurs restantes se concentrent sur les **cas négatifs** (codes devinés) et les **emplois multiples**. Changement suivant : écrire des **règles de décision** explicites.

In [ ]:
REGLES = """
RÈGLES
1. Si la personne exerce plusieurs activités, code uniquement l'emploi principal (le premier cité).
2. Si la description est trop vague pour déterminer un groupe de base à 4 chiffres, réponds "NON_CODABLE" dans le champ code_citp. Ne devine jamais."""

SYSTEME_V3 = SYSTEME_V1 + "\n" + REGLES + "\n" + FORMAT_JSON

def messages_v3(texte):
    return [{"role": "system", "content": SYSTEME_V3},
            {"role": "user", "content": f"Description : {texte}"}]

df_v3, res_v3 = evaluer("v3", messages_v3, json_mode=True, changement="+ règles : emploi principal, NON_CODABLE")
afficher_erreurs(df_v3, 10)

Il reste des confusions entre **catégories proches** : vente au marché (5211), vente ambulante de nourriture (5212) et vente ambulante d’autres produits (9520). Une règle écrite serait possible ; ici, nous testons l’autre levier : **montrer des exemples**, tirés du réservoir (jamais du jeu d’évaluation).

In [ ]:
TEXTE_EXEMPLES = "\nEXEMPLES\n" + "\n".join(f'- "{t}" → {c}' for t, c in RESERVOIR_EXEMPLES)
SYSTEME_V4 = SYSTEME_V1 + "\n" + REGLES + "\n" + TEXTE_EXEMPLES + "\n" + FORMAT_JSON

def messages_v4(texte):
    return [{"role": "system", "content": SYSTEME_V4},
            {"role": "user", "content": f"Description : {texte}"}]

df_v4, res_v4 = evaluer("v4", messages_v4, json_mode=True, changement="+ 6 exemples du réservoir")
atteinte_v4 = bool(res_v4["exactitude_stricte"] >= CIBLE_EXACTITUDE)
encadre(f"Cible de {CIBLE_EXACTITUDE:.0%} {'atteinte' if atteinte_v4 else '<b>pas encore atteinte</b>'} "
        f"avec la v4 ({res_v4['exactitude_stricte']:.0%}). "
        + ("" if atteinte_v4 else "Vous tenterez de la dépasser en section 7."), "note" if atteinte_v4 else "attention")
afficher_erreurs(df_v4, 10)
print("\nPrompt système v4 :\n" + "─" * 60 + "\n" + SYSTEME_V4)

### 3.5 Le piège de la fuite : « améliorer » en recopiant le test

Tentation classique : copier dans le prompt les cas du jeu d’évaluation qui échouent. Le score monte… mais le prompt a-t-il vraiment progressé ? Vérifions sur le **jeu réservé**, que le prompt n’a jamais vu.

In [ ]:
ids_fuites = set(df_v4.loc[~df_v4["correct_strict"], "id"]) | set(JEU_EVAL.loc[JEU_EVAL["type_cas"] != "typique", "id"])
cas_fuites = JEU_EVAL[JEU_EVAL["id"].isin(ids_fuites)]   # les cas en échec ET les cas difficiles
TEXTE_FUITE = TEXTE_EXEMPLES + "\n" + "\n".join(f'- "{t}" → {c}' for t, c in zip(cas_fuites["texte"], cas_fuites["code_ref"]))
SYSTEME_V5 = SYSTEME_V1 + "\n" + REGLES + "\n" + TEXTE_FUITE + "\n" + FORMAT_JSON

def messages_v5(texte):
    return [{"role": "system", "content": SYSTEME_V5},
            {"role": "user", "content": f"Description : {texte}"}]

df_v5, res_v5 = evaluer("v5", messages_v5, json_mode=True, changement="+ cas du jeu d'évaluation copiés (FUITE)")
print("Contrôle sur le jeu réservé :")
_, res_v4_h = evaluer("v4", messages_v4, jeu=JEU_RESERVE, json_mode=True, changement="contrôle")
_, res_v5_h = evaluer("v5", messages_v5, jeu=JEU_RESERVE, json_mode=True, changement="contrôle")

In [ ]:
comparaison = pd.DataFrame({
    "Jeu d'évaluation": [res_v4["exactitude_stricte"], res_v5["exactitude_stricte"]],
    "Jeu réservé": [res_v4_h["exactitude_stricte"], res_v5_h["exactitude_stricte"]],
}, index=["v4 (exemples du réservoir)", "v5 (cas du test copiés)"])
ax = comparaison.plot.bar(color=[VERT_FONCE, OCRE], figsize=(8, 3.4), rot=0, width=0.65)
for c in ax.containers:
    ax.bar_label(c, labels=[f"{v:.0%}" for v in c.datavalues], padding=2)
ax.set_ylim(0, 1.15); ax.set_yticks([]); ax.set_title("La fuite gonfle le score… sans gain réel")
ax.legend(frameon=False, loc="upper center", bbox_to_anchor=(0.5, -0.12), ncol=2)
plt.tight_layout(); plt.show()
decider("v5", "rejeté : fuite")
encadre("<b>Leçon :</b> la v5 « gagne » sur le jeu d'évaluation, mais pas sur le jeu réservé. "
        "Elle a mémorisé le test. Les exemples du prompt doivent toujours venir d'un réservoir distinct.", "risque")

### 3.6 Le journal des itérations

Le journal rend votre prompt **auditable** : chaque version, un seul changement, un score, une décision.

In [ ]:
decider("v1", "retenu"); decider("v2", "retenu"); decider("v3", "retenu"); decider("v4", "retenu")

def afficher_journal():
    j = pd.DataFrame([x for x in JOURNAL if x["jeu"] == "évaluation"])
    colonnes = ["version", "changement", "exactitude_stricte", "exactitude_souple", "json_valide",
                "exact_typique", "exact_limite", "exact_négatif", "jetons_par_cas", "decision"]
    vue = j[colonnes].copy()
    for c in ["exactitude_stricte", "exactitude_souple", "json_valide", "exact_typique", "exact_limite", "exact_négatif"]:
        vue[c] = vue[c].map(lambda x: f"{x:.0%}")
    display(vue.set_index("version"))
    fig, ax = plt.subplots(figsize=(10, 3.8))
    x = range(len(j))
    couleurs = [BRIQUE if str(d).startswith("rejet") else VERT_FONCE for d in j["decision"]]
    ax.bar([i - 0.2 for i in x], j["exactitude_stricte"], 0.4, color=couleurs, label="Exactitude stricte")
    ax.bar([i + 0.2 for i in x], j["json_valide"], 0.4, color=RAMPE[0], label="JSON valide")
    for i, v in enumerate(j["exactitude_stricte"]):
        ax.text(i - 0.2, v + 0.02, f"{v:.0%}", ha="center", fontsize=9, color=ENCRE)
    ax.axhline(CIBLE_EXACTITUDE, color=OR, ls="--", lw=1.5)
    ax.text(-0.45, CIBLE_EXACTITUDE + 0.02, f"cible {CIBLE_EXACTITUDE:.0%}", color=OCRE, ha="left", fontweight="bold")
    ax.set_xticks(list(x)); ax.set_ylim(0, 1.12); ax.set_yticks([])
    ax.set_xticklabels([f"{v}\n{'rejetée' if str(d).startswith('rejet') else ''}" for v, d in zip(j["version"], j["decision"])])
    ax.legend(frameon=False, loc="upper center", bbox_to_anchor=(0.5, -0.18), ncol=2)
    ax.set_title("Historique des versions sur le jeu d'évaluation (rouge = version rejetée)")
    plt.tight_layout(); plt.show()

afficher_journal()

### 3.7 Au-delà de la moyenne : l’analyse par classe

Une exactitude globale élevée peut cacher une catégorie **systématiquement** mal codée. Regardons la matrice de confusion de la meilleure version légitime (v4).

In [ ]:
from sklearn.metrics import classification_report

rapport = classification_report(df_v4["code_ref"], df_v4["code_obtenu"].fillna("AUCUN"),
                                 output_dict=True, zero_division=0)
par_classe = (pd.DataFrame(rapport).T.drop(["accuracy", "macro avg", "weighted avg"], errors="ignore")
              .query("support > 0")[["precision", "recall", "support"]]
              .rename(columns={"precision": "précision", "recall": "rappel", "support": "effectif"}))
display(par_classe.sort_values("rappel").head(8).style.format({"précision": "{:.0%}", "rappel": "{:.0%}", "effectif": "{:.0f}"})
        .background_gradient(subset=["rappel"], cmap="RdYlGn", vmin=0, vmax=1))

confusions = df_v4[df_v4["code_ref"] != df_v4["code_obtenu"]]
if len(confusions):
    display(pd.crosstab(confusions["code_ref"], confusions["code_obtenu"].fillna("AUCUN"))
            .rename_axis(index="attendu", columns="obtenu"))
encadre("Les confusions restantes indiquent <b>où</b> agir : une règle ciblée ou un exemple supplémentaire "
        "pour la catégorie concernée. Vous le ferez vous-même en section 7.", "note")

<div style="background:#0E7C86;color:#FFFFFF;padding:10px 18px;border-radius:8px">
<b>3.8 Aparté : utiliser un LLM comme juge</b>
</div>

Pour la **rédaction** ou les **questions-réponses**, aucun script ne peut noter automatiquement. On peut confier la notation à un second modèle, le **juge**, à partir d’une grille écrite. Mais le juge a des **biais connus** (Zheng et al., 2023) : il favorise souvent la réponse présentée **en premier**, les réponses **plus longues** et celles qui ressemblent à **son propre style**.

Test simple du **biais de position** : on présente la même paire dans les deux ordres. Un juge fiable doit désigner le même résumé.

In [ ]:
SOURCE_JUGE = ("Note méthodologique : l'enquête EPA 2025 a interrogé 12 480 ménages entre février et avril, "
               "avec un taux de réponse de 91 %. Les résultats sont représentatifs au niveau régional.")
PAIRES = [
    ("L'EPA 2025 couvre 12 480 ménages (février-avril, réponse 91 %), représentative par région.",
     "L'enquête EPA 2025, conduite au cours du premier semestre, a interrogé un large échantillon de ménages "
     "dans toutes les régions du pays, avec un excellent taux de réponse, ce qui garantit des résultats "
     "robustes et utiles pour les politiques publiques."),
    ("Enquête de 12 480 ménages, taux de réponse 91 %.",
     "L'EPA 2025 porte sur 12 480 ménages interrogés de février à avril ; le taux de réponse atteint 91 % "
     "et les résultats sont représentatifs au niveau régional."),
]
SYSTEME_JUGE = ("Tu es un ÉVALUATEUR de résumés statistiques. Compare deux résumés d'une même source selon la grille : "
                "exactitude des chiffres (priorité), complétude, concision. "
                'Réponds en JSON : {"meilleur": 1 ou 2, "justification": "..."}')

def juger(source, r1, r2):
    contenu = f"SOURCE :\n{source}\n\nRÉSUMÉ 1 :\n{r1}\n\nRÉSUMÉ 2 :\n{r2}"
    rep = LLM.chat([{"role": "system", "content": SYSTEME_JUGE}, {"role": "user", "content": contenu}],
                   json_mode=True, etiquette="juge")
    try:
        return int(json.loads(rep["texte"])["meilleur"])
    except Exception:
        return None

lignes = []
for i, (a, b) in enumerate(PAIRES, 1):
    ordre_ab = juger(SOURCE_JUGE, a, b)            # 1 = A
    ordre_ba = juger(SOURCE_JUGE, b, a)            # 2 = A
    choix_ab = "A" if ordre_ab == 1 else "B"
    choix_ba = "A" if ordre_ba == 2 else "B"
    lignes.append(dict(paire=i, choix_ordre_AB=choix_ab, choix_ordre_BA=choix_ba,
                       coherent="✅" if choix_ab == choix_ba else "❌ biais de position"))
display(pd.DataFrame(lignes))
encadre("<b>Calibrer avant de faire confiance :</b> faites noter 20 à 30 sorties par des humains, comparez avec le juge, "
        "présentez chaque paire dans les deux ordres, et n'utilisez le juge que si l'accord est élevé.", "attention")

<div style="background:#D49A00;color:#FFFFFF;padding:14px 22px;border-radius:10px;border-left:10px solid #00553A">
<span style="color:#FFFFFF;font-size:30px;font-weight:800">04</span>&nbsp;&nbsp;<span style="font-size:22px;font-weight:700">Maîtriser la variance</span><br>
<i style="color:#FFF7E0">Même prompt, même entrée : pourquoi une autre réponse ?</i>
</div>

### 4.1 D’où vient la variance ?

| Source | Explication | Levier |
|---|---|---|
| 🎲 **Échantillonnage** | le modèle choisit chaque jeton avec une part d’aléa réglée par la **température** | température basse (0 à 0,2) pour extraire et coder |
| ❓ **Consignes ambiguës** | une règle floue est tranchée différemment à chaque exécution | règles d’arbitrage explicites |
| 🔄 **Mises à jour silencieuses** | un alias générique peut pointer vers une nouvelle version | figer l’identifiant exact du modèle |
| 📄 **Variation des entrées** | langue, mise en page, fautes | jeu d’évaluation représentatif |

### 4.2 Le test de cohérence en 5 exécutions

On exécute chaque cas **5 fois** et on mesure le **taux d’accord** : la part des exécutions qui donnent la réponse la plus fréquente. Le cache est désactivé pour ce test, sinon les répétitions à température 0 seraient trivialement identiques.

In [ ]:
SOUS_JEU = JEU_EVAL[JEU_EVAL["id"].isin(["E01", "E02", "E05", "E13", "L01", "L02", "L03", "L04", "L06", "N01"])]
N_REPETITIONS = 5

def test_coherence(construire_messages, temperature, etiquette):
    lignes = []
    for cas in SOUS_JEU.itertuples():
        codes = []
        for _ in range(N_REPETITIONS):
            r = LLM.chat(construire_messages(cas.texte), temperature=temperature, json_mode=True,
                         utiliser_cache=False, etiquette=etiquette)
            strict, souple, _ = extraire_code(r["texte"])
            codes.append(strict or souple or "ILLISIBLE")
        majoritaire, n = Counter(codes).most_common(1)[0]
        lignes.append(dict(id=cas.id, texte=cas.texte[:38], code_ref=cas.code_ref, reponses=" ".join(codes),
                           accord=n / N_REPETITIONS, exact_1er_essai=codes[0] == cas.code_ref,
                           exact_vote=majoritaire == cas.code_ref))
    return pd.DataFrame(lignes)

coh_t0 = test_coherence(messages_v3, 0.0, "variance_t0")
coh_t1 = test_coherence(messages_v3, 1.0, "variance_t1")
display(coh_t1[["id", "texte", "code_ref", "reponses", "accord"]].style.format({"accord": "{:.0%}"})
        .background_gradient(subset=["accord"], cmap="RdYlGn", vmin=0.4, vmax=1))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3.4))
x = range(len(coh_t0))
ax.bar([i - 0.2 for i in x], coh_t0["accord"], 0.4, color=VERT_FONCE, label="température 0")
ax.bar([i + 0.2 for i in x], coh_t1["accord"], 0.4, color=OCRE, label="température 1")
ax.set_xticks(list(x)); ax.set_xticklabels(coh_t0["id"]); ax.set_ylim(0, 1.15)
ax.set_ylabel("taux d'accord sur 5 exécutions"); ax.legend(frameon=False, ncol=2, loc="upper left")
ax.set_title("Les cas limites (L..) sont les plus instables quand la température monte")
plt.tight_layout(); plt.show()
print(f"Accord moyen · température 0 : {coh_t0['accord'].mean():.0%} · température 1 : {coh_t1['accord'].mean():.0%}")

<div style="background:#FBF1D9;border-left:5px solid #D49A00;padding:12px 16px;border-radius:6px">
⚠️ <b>Une température de 0 réduit l’aléa ; elle ne garantit pas des sorties identiques</b> chez tous les fournisseurs (en simulation, l’accord est parfait à température 0 ; ce n’est pas toujours le cas avec un vrai modèle). <b>Mesurez toujours</b> l’accord sur votre jeu.
</div>

### 4.3 Le vote majoritaire (auto-cohérence)

Pour les cas à fort enjeu, on peut exécuter plusieurs fois et retenir la **réponse majoritaire** (Wang et al., 2022). C’est plus coûteux, mais un **désaccord entre exécutions** est aussi un excellent signal pour **orienter le cas vers un codificateur humain**.

In [ ]:
recap = pd.DataFrame({
    "Exactitude, 1 seule exécution": [coh_t1["exact_1er_essai"].mean()],
    "Exactitude, vote sur 5": [coh_t1["exact_vote"].mean()],
    "Cas à revoir (accord < 100 %)": [(coh_t1["accord"] < 1).sum()],
    "Coût relatif": ["× 5"],
}, index=["température 1"])
display(recap.style.format({"Exactitude, 1 seule exécution": "{:.0%}", "Exactitude, vote sur 5": "{:.0%}"}))
a_revoir = coh_t1[coh_t1["accord"] < 1]["id"].tolist()
encadre(f"Cas à router vers un codificateur humain : <b>{', '.join(a_revoir) or 'aucun'}</b>. "
        "Le désaccord entre exécutions est un indicateur de confiance gratuit.", "retenir")

### 4.4 Six leviers pour des sorties reproductibles

| Paramètres du modèle | Conception du prompt |
|---|---|
| 🌡️ **Température basse** (0 à 0,2) pour extraire et coder | 🧾 **Contraindre le format** : schéma JSON, liste fermée de valeurs |
| 🏷️ **Figer la version du modèle** et la noter au journal | ⚖️ **Règles d’arbitrage** : « si deux emplois, coder le principal » |
| ✂️ **Plafonner la sortie** (`max_tokens`, limite de mots) | 🗳️ **Voter sur les cas difficiles**, désaccords vers un humain |

In [ ]:
# 🏷️ Toujours consigner la configuration exacte utilisée
CONFIGURATION = dict(fournisseur=FOURNISSEUR_ACTIF, modele=MODELE_ACTIF, temperature=0.0,
                     json_mode=True, prompt_version="v4", jeu_eval=EMPREINTE_EVAL,
                     date=pd.Timestamp.now().strftime("%Y-%m-%d %H:%M"))
display(pd.Series(CONFIGURATION, name="configuration").to_frame())
if MODELE_ACTIF and not re.search(r"\d", MODELE_ACTIF):
    encadre("Le nom du modèle ne contient pas de numéro de version : vérifiez qu'il ne s'agit pas d'un alias mouvant.", "attention")

<div style="background:#C4621D;color:#FFFFFF;padding:14px 22px;border-radius:10px;border-left:10px solid #F5C242">
<span style="color:#F5C242;font-size:30px;font-weight:800">05</span>&nbsp;&nbsp;<span style="font-size:22px;font-weight:700">Contexte, jetons et coûts</span><br>
<i style="color:#FBE9DC">Combien coûterait la codification de 100 000 réponses d’enquête ?</i>
</div>

### 5.1 Les jetons : l’unité du contexte et du coût

Les modèles lisent et écrivent des **jetons** (*tokens*), pas des mots. Règle empirique d’OpenAI pour l’anglais : **1 jeton ≈ 4 caractères ≈ ¾ de mot**. Beaucoup d’autres langues demandent **plus de jetons** pour le même sens (Petrov et al., 2023), ce qui pèse directement sur le budget des INS travaillant en français, portugais, arabe ou langues africaines.

> **Coût = jetons d’entrée × prix d’entrée + jetons de sortie × prix de sortie**
> **Fenêtre de contexte ≥ consignes + exemples + document + réponse**

In [ ]:
phrases = {
    "Anglais": "The consumer price index rose by 2.4 percent in June compared with the previous month.",
    "Français": "L'indice des prix à la consommation a augmenté de 2,4 % en juin par rapport au mois précédent.",
    "Portugais": "O índice de preços ao consumidor subiu 2,4 % em junho em relação ao mês anterior.",
    "Kiswahili": "Fahirisi ya bei za bidhaa kwa mlaji iliongezeka kwa asilimia 2.4 mwezi Juni ikilinganishwa na mwezi uliopita.",
    "Arabe": "ارتفع مؤشر أسعار المستهلك بنسبة 2.4 في المائة في يونيو مقارنة بالشهر السابق.",
}
jetons = pd.Series({langue: compter_jetons(p) for langue, p in phrases.items()}).sort_values()
ax = jetons.plot.barh(color=[RAMPE[2 + i] for i in range(len(jetons))], figsize=(8, 3))
ax.bar_label(ax.containers[0], labels=[f"{v} jetons (× {v / jetons['Anglais']:.2f})" for v in jetons], padding=3)
ax.set_xlim(0, jetons.max() * 1.45); ax.set_xticks([])
ax.set_title("La même phrase, un nombre de jetons différent selon la langue")
plt.tight_layout(); plt.show()
print("Méthode :", METHODE_JETONS, "· traductions fournies à titre d'illustration")
if "approximation" in METHODE_JETONS:
    encadre("tiktoken est indisponible : l'approximation par caractères ne reflète pas les différences réelles entre langues. "
            "Sur Colab ou Kaggle, relancez avec tiktoken pour voir l'écart.", "attention")

### 5.2 Où vont les jetons de nos prompts ?

Chaque appel envoie une **partie fixe** (consignes, liste, exemples) et une **partie variable** (la description). Voyons comment les versions ont fait grossir la partie fixe.

In [ ]:
versions = {"v0": messages_v0, "v1": messages_v1, "v2": messages_v2, "v3": messages_v3, "v4": messages_v4, "v5": messages_v5}
exemple = JEU_EVAL["texte"].iloc[0]
taille = pd.DataFrame({
    v: {"fixe": sum(compter_jetons(m["content"]) for m in f(exemple) if m["role"] == "system"),
        "variable": sum(compter_jetons(m["content"]) for m in f(exemple) if m["role"] == "user")}
    for v, f in versions.items()}).T
ax = taille.plot.bar(stacked=True, color=[VERT_FONCE, TERRE], figsize=(9, 3.4), rot=0, width=0.6)
for i, total in enumerate(taille.sum(axis=1)):
    ax.text(i, total + 10, f"{total:.0f}", ha="center", color=ENCRE)
ax.set_ylabel("jetons d'entrée par appel"); ax.legend(["partie fixe (système)", "partie variable (cas)"], frameon=False)
ax.set_title("La qualité a un prix : la partie fixe domine")
plt.tight_layout(); plt.show()
display(taille)

### 5.3 Calculateur de coût : exemple chiffré

Codifier **100 000** descriptions avec la v4. Les prix sont **illustratifs** : remplacez-les par le tarif réel dans la cellule de configuration. Quatre scénarios :
1. **Base** : un appel par description ;
2. **+ cache de prompt** : la partie fixe, identique d’un appel à l’autre, est facturée avec une remise chez le fournisseur (hypothèse : 90 %) ;
3. **+ lots** : 20 descriptions par appel, la partie fixe n’est envoyée qu’une fois par lot ;
4. **les deux combinés**.

In [ ]:
def scenarios_cout(n_docs, fixe, variable, sortie, remise_cache=0.90, taille_lot=20):
    pe, ps = PRIX_ENTREE_PAR_MILLION / 1e6, PRIX_SORTIE_PAR_MILLION / 1e6
    appels_lot = math.ceil(n_docs / taille_lot)
    base = n_docs * (fixe + variable) * pe + n_docs * sortie * ps
    cache = n_docs * fixe * pe * (1 - remise_cache) + n_docs * variable * pe + n_docs * sortie * ps
    lots = (appels_lot * fixe + n_docs * variable) * pe + n_docs * sortie * ps
    les_deux = (appels_lot * fixe * pe * (1 - remise_cache)) + n_docs * variable * pe + n_docs * sortie * ps
    return pd.Series({"Base": base, "+ cache de prompt": cache, f"+ lots ({taille_lot}/appel)": lots,
                      "cache + lots": les_deux})

# Hypothèses de l'exemple de la présentation
s = scenarios_cout(100_000, fixe=2_500, variable=50, sortie=40)
ax = s.plot.barh(color=["#A9B5B0", RAMPE[2], RAMPE[4], RAMPE[6]], figsize=(9, 3.2))
ax.bar_label(ax.containers[0], labels=[f"{v:,.2f} $".replace(",", " ") for v in s], padding=4)
ax.invert_yaxis(); ax.set_xlim(0, s.max() * 1.25); ax.set_xticks([])
ax.set_title("Coût pour 100 000 descriptions (prix illustratifs)")
plt.tight_layout(); plt.show()
print(f"Gain de la conception du prompt : ÷ {s.iloc[0] / s.iloc[-1]:.0f}")

In [ ]:
# 🎛️ Calculateur interactif (si ipywidgets est disponible) avec les jetons réels de VOTRE v4
fixe_v4, variable_v4 = int(taille.loc["v4", "fixe"]), int(taille.loc["v4", "variable"])
try:
    import ipywidgets as w
    def _calcul(n_docs, taille_lot, remise_cache, sortie):
        r = scenarios_cout(n_docs, fixe_v4, variable_v4, sortie, remise_cache / 100, taille_lot)
        display(r.map(lambda v: f"{v:,.2f} $").to_frame("coût estimé"))
    w.interact(_calcul,
               n_docs=w.IntSlider(value=100_000, min=1_000, max=1_000_000, step=1_000, description="documents"),
               taille_lot=w.IntSlider(value=20, min=1, max=50, description="taille lot"),
               remise_cache=w.IntSlider(value=90, min=0, max=100, step=5, description="remise %"),
               sortie=w.IntSlider(value=40, min=5, max=500, step=5, description="jetons sortie"))
except Exception as e:
    print(f"Widgets indisponibles ({type(e).__name__}) : version statique avec vos jetons v4.")
    display(scenarios_cout(100_000, fixe_v4, variable_v4, 40).map(lambda v: f"{v:,.2f} $").to_frame("coût estimé"))

### 5.4 Le traitement par lots, en vrai

On envoie maintenant les 30 cas en **3 appels de 10 cas** au lieu de 30 appels. On compare les **jetons réellement consommés** et **l’exactitude** : un lot plus gros coûte moins cher, mais la qualité peut baisser quand beaucoup d’éléments partagent un même appel.

In [ ]:
FORMAT_LOT = """
FORMAT DE SORTIE
Tu reçois plusieurs descriptions, chacune précédée de son identifiant entre crochets.
Réponds uniquement avec un objet JSON :
{"resultats": [{"id": "<identifiant>", "code_citp": "<code à 4 chiffres ou NON_CODABLE>"}]}"""
SYSTEME_LOT = SYSTEME_V1 + "\n" + REGLES + "\n" + TEXTE_EXEMPLES + "\n" + FORMAT_LOT

def evaluer_par_lots(taille_lot=10, etiquette="lots"):
    obtenus = {}
    for debut in range(0, len(JEU_EVAL), taille_lot):
        bloc = JEU_EVAL.iloc[debut:debut + taille_lot]
        contenu = "\n".join(f"[{c.id}] {c.texte}" for c in bloc.itertuples())
        r = LLM.chat([{"role": "system", "content": SYSTEME_LOT}, {"role": "user", "content": contenu}],
                     json_mode=True, max_tokens=2048, etiquette=etiquette)
        try:
            for item in json.loads(r["texte"])["resultats"]:
                obtenus[str(item["id"])] = str(item["code_citp"]).upper()
        except Exception:
            pass
    return (JEU_EVAL["id"].map(obtenus) == JEU_EVAL["code_ref"]).mean()

exact_lots = evaluer_par_lots(10, "lots_10")
b = LLM.bilan()
comparatif = pd.DataFrame({
    "appels": [len(JEU_EVAL), b.loc["lots_10", "appels"]],
    "jetons d'entrée": [df_v4["jetons_entree"].sum(), b.loc["lots_10", "jetons_entree"]],
    "exactitude": [res_v4["exactitude_stricte"], exact_lots],
}, index=["1 cas par appel (v4)", "10 cas par appel"])
display(comparatif.style.format({"exactitude": "{:.0%}", "jetons d'entrée": "{:,.0f}", "appels": "{:.0f}"}))
encadre("Le regroupement divise les jetons d'entrée, mais <b>re-testez toujours l'exactitude</b> : "
        "au-delà d'une certaine taille de lot, le modèle applique moins bien les consignes et les exemples.", "attention")

### 5.5 Deux types de mise en cache

| | **Cache de prompt** (chez le fournisseur) | **Cache de réponses** (chez vous) |
|---|---|---|
| Principe | le début identique du prompt n’est pas recalculé | la réponse d’une requête identique est réutilisée |
| Gain | latence et prix réduits sur la partie fixe | appel gratuit, instantané, reproductible |
| Condition | **contenu stable d’abord, variable à la fin** | même modèle, même prompt, même entrée, température 0 |
| Bonus | — | **piste d’audit** pour chaque chiffre publié |

Notre client applique le **cache de réponses**. Relançons exactement la même évaluation :

In [ ]:
LLM.cache.clear()   # on repart d'un cache vide pour la démonstration
t0 = time.perf_counter(); _ = evaluer("v4", messages_v4, json_mode=True, afficher=False, consigner=False); duree_1 = time.perf_counter() - t0
t0 = time.perf_counter(); _ = evaluer("v4", messages_v4, json_mode=True, afficher=False, consigner=False); duree_2 = time.perf_counter() - t0
derniers = pd.DataFrame(LLM.appels[-60:])
print(f"1re exécution : {duree_1:.2f} s · 2e exécution : {duree_2:.3f} s"
      + ("  (en simulation, la 1re exécution est déjà quasi instantanée)" if MODE_SIMULATION else ""))
print(f"Appels servis depuis le cache lors de la 2e exécution : {derniers.tail(30)['depuis_cache'].sum()} / 30")
en_cache_fournisseur = pd.DataFrame(LLM.appels)["jetons_en_cache_fournisseur"].sum()
print(f"Jetons signalés en cache par le fournisseur (si pris en charge) : {en_cache_fournisseur:,}")

<div style="background:#F6E3E0;border-left:5px solid #B83B2E;padding:12px 16px;border-radius:6px">
⛔ <b>Erreur fréquente qui casse le cache de prompt :</b> placer une date, un identifiant ou le cas à coder <b>au début</b> du prompt. Un seul caractère différent au début suffit à invalider tout le reste. Structure recommandée : <code>[consignes · liste · exemples]</code> puis <code>[cas]</code>, exactement ce que font nos versions v1 à v4.
</div>

### 5.6 « Perdu au milieu » : la position de l’information compte

Liu et al. (2023) ont montré que les modèles exploitent mieux une information placée **au début ou à la fin** d’un long contexte qu’**au milieu**. Testons-le : un long bulletin fictif contient le taux d’inflation de mars, placé au début, au milieu ou à la fin, et un **leurre** (le taux de février).

In [ ]:
def bulletin(position, valeur, n_paragraphes=40):
    regions = ["Nord", "Sud", "Est", "Ouest", "Centre", "Littoral", "Plateaux", "Savanes"]
    filler = [f"Dans la région {regions[i % 8]}, l'indicateur {i} de l'activité économique a évolué de "
              f"{(i * 7) % 11 + 0.5:.1f} points au cours de la période, selon les relevés administratifs habituels "
              f"transmis par les services déconcentrés et consolidés par la direction des synthèses."
              for i in range(n_paragraphes)]
    cible = f"Au niveau national, le taux d'inflation annuel du mois de mars s'établit à {valeur} %."
    leurre = "Pour rappel, le taux d'inflation annuel du mois de février s'établit à 4,1 %."
    idx = {"début": 0, "milieu": n_paragraphes // 2, "fin": n_paragraphes}[position]
    filler.insert(n_paragraphes // 2 + 5 if position != "milieu" else 3, leurre)
    filler.insert(idx if position != "fin" else len(filler), cible)
    return "BULLETIN MENSUEL DE CONJONCTURE (fictif)\n\n" + "\n".join(filler)

resultats_aiguille = []
for position in ["début", "milieu", "fin"]:
    for valeur in ["7,3", "5,8", "6,2"]:
        doc = bulletin(position, valeur)
        r = LLM.chat([{"role": "system", "content": "Réponds uniquement à partir du document, par la valeur avec son unité."},
                      {"role": "user", "content": doc + "\n\nQuestion : quel est le taux d'inflation annuel du mois de mars ?"}],
                     max_tokens=512, etiquette="aiguille")
        resultats_aiguille.append(dict(position=position, valeur=valeur, correct=valeur in r["texte"],
                                       jetons=compter_jetons(doc)))
ra = pd.DataFrame(resultats_aiguille)
score_pos = ra.groupby("position", sort=False)["correct"].mean()
fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(range(3), score_pos.values, marker="o", color=TERRE, lw=3, ms=10)
for i, v in enumerate(score_pos.values):
    ax.text(i, v + 0.07, f"{v:.0%}", ha="center", color=ENCRE, fontweight="bold")
ax.set_xticks(range(3)); ax.set_xticklabels([f"information au {p}" if p != "fin" else "information à la fin" for p in score_pos.index])
ax.set_ylim(-0.05, 1.2); ax.set_yticks([]); ax.set_xlim(-0.3, 2.3)
ax.set_title(f"Information cherchée selon sa position (~{ra['jetons'].mean():.0f} jetons de contexte)")
plt.tight_layout(); plt.show()

<div style="background:#F4F7F5;border-left:5px solid #00704A;padding:12px 16px;border-radius:6px">
📌 <b>Moins de contexte, mieux placé.</b> Les modèles récents résistent mieux à cet effet (vous le constaterez peut-être en mode réel), mais la règle reste valable, ne serait-ce que pour le <b>coût</b> :
<ul>
<li><b>n’envoyer que le pertinent</b> : les 3 pages du tableau, pas l’annuaire de 120 pages (c’est le rôle du RAG) ;</li>
<li><b>résumer ou découper</b> les longs documents ;</li>
<li><b>placer les consignes clés aux extrémités</b> et rappeler la règle essentielle à la fin ;</li>
<li><b>limiter la réponse</b> (limite de mots, <code>max_tokens</code>).</li>
</ul>
</div>

<div style="background:#B83B2E;color:#FFFFFF;padding:14px 22px;border-radius:10px;border-left:10px solid #F5C242">
<span style="color:#F5C242;font-size:30px;font-weight:800">06</span>&nbsp;&nbsp;<span style="font-size:22px;font-weight:700">Prompt, RAG ou affinage ?</span><br>
<i style="color:#F8E1DE">Quel outil corrige quel échec ?</i>
</div>

### 6.1 Trois outils, trois problèmes

| | ✏️ **Prompt** *(commencer ici)* | 📚 **RAG** *(ajouter du savoir)* | ⚙️ **Affinage** *(modifier le modèle)* |
|---|---|---|---|
| **Corrige** | un comportement descriptible en mots | un savoir manquant, changeant ou à citer | un comportement constant sur une tâche étroite à fort volume |
| **Données requises** | quelques exemples | un corpus documentaire | des centaines à milliers de paires annotées |
| **1er résultat** | heures | jours | semaines |
| **Coût** | faible | moyen (embeddings, index) | élevé (entraînement, hébergement) |
| **Exemple INS** | rédiger un communiqué | Q&R sur les notes méthodologiques | codification CITP nationale avec un petit modèle interne |

*Durées et volumes : ordres de grandeur indicatifs.*

### 6.2 Démonstration : quand le prompt ne suffit pas

Question sur une enquête **fictive** (République de Fictivia) : le modèle ne peut pas connaître la réponse. Sans documents, il risque d’**halluciner** un chiffre plausible.

In [ ]:
NOTE_METHODO = [
    "P1. L'Enquête sur la population active (EPA) 2025 de la République de Fictivia est conduite par l'Institut national de la statistique de Fictivia.",
    "P2. La période de collecte s'étend du 3 février au 30 avril 2025, par entretiens assistés par tablette.",
    "P3. L'échantillon comprend 12 480 ménages répartis dans les 14 régions du pays, tirés selon un plan stratifié à deux degrés.",
    "P4. Le taux de réponse global atteint 91 %, avec un minimum de 84 % dans la région du Littoral.",
    "P5. Les professions sont codées selon la CITP-08 au niveau des groupes de base à quatre chiffres.",
    "P6. Les pondérations sont calées sur les projections démographiques de 2025 par sexe, âge et milieu de résidence.",
    "P7. Les résultats sont représentatifs au niveau national, régional et par milieu urbain ou rural.",
    "P8. Les microdonnées anonymisées sont accessibles aux chercheurs sur demande, après signature d'un accord de confidentialité.",
]
QUESTION = "Quelle est la taille de l'échantillon de l'EPA 2025 de Fictivia ?"

sans_rag = LLM.chat([{"role": "user", "content": QUESTION}], etiquette="rag")
print("❌ Sans documents :", sans_rag["texte"])

In [ ]:
# 📚 Mini-RAG : recherche TF-IDF des passages pertinents, puis réponse ancrée avec citation
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# n-grammes de caractères : plus robustes aux variations du français (l'échantillon / échantillons…)
vectoriseur = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5)).fit(NOTE_METHODO)
similarites = cosine_similarity(vectoriseur.transform([QUESTION]), vectoriseur.transform(NOTE_METHODO))[0]
top = similarites.argsort()[::-1][:2]
passages = [NOTE_METHODO[i] for i in top]
display(pd.DataFrame({"passage": [NOTE_METHODO[i][:90] + "…" for i in top], "similarité": similarites[top].round(3)}))

SYSTEME_RAG = ("Réponds uniquement à partir des passages fournis et cite le passage utilisé entre crochets, par ex. [P3]. "
               "Si l'information n'y figure pas, réponds : « Information absente des documents ».")
avec_rag = LLM.chat([{"role": "system", "content": SYSTEME_RAG},
                     {"role": "user", "content": "PASSAGES :\n" + "\n".join(passages) + f"\n\nQUESTION : {QUESTION}"}],
                    etiquette="rag")
print("✅ Avec RAG :", avec_rag["texte"])
print("Réponse ancrée dans la source :", "12 480" in avec_rag["texte"])

jetons_tout = compter_jetons("\n".join(NOTE_METHODO)); jetons_top = compter_jetons("\n".join(passages))
print(f"Contexte envoyé : {jetons_top} jetons au lieu de {jetons_tout} pour la note entière (÷ {jetons_tout / jetons_top:.1f}). "
      "Sur un annuaire de 120 pages, l'écart est bien plus grand.")

### 6.3 Arbre de décision guidé par vos résultats

Répondez aux trois questions : l’outil recommandé s’affiche.

In [ ]:
def recommander(cible_atteinte, echecs_factuels, constance_a_grande_echelle, donnees_confidentielles):
    if cible_atteinte:
        reco, couleur = "✅ Déployer, surveiller, re-tester à chaque mise à jour du modèle.", VERT
    elif echecs_factuels:
        reco, couleur = "📚 Ajouter une recherche documentaire (RAG) et exiger des citations.", SARCELLE
    elif constance_a_grande_echelle:
        reco, couleur = "⚙️ Envisager l'affinage d'un modèle ouvert, avec des centaines d'exemples annotés.", BRIQUE
    else:
        reco, couleur = "🔗 Décomposer la tâche en une chaîne de prompts, ou collecter plus d'exemples annotés.", OCRE
    souverainete = ("<br>🔒 <b>Données confidentielles :</b> seuls un fournisseur sous votre contrôle juridique "
                    "ou un modèle ouvert hébergé en interne (par ex. via Ollama) sont éligibles.") if donnees_confidentielles else ""
    display(HTML(f'<div style="border-left:6px solid {couleur};background:#F4F7F5;padding:12px 16px;border-radius:6px">'
                 f'<b>{reco}</b>{souverainete}<br><i>Quelle que soit la voie : relancer le même jeu d\'évaluation.</i></div>'))

meilleur = max((j for j in JOURNAL if j["jeu"] == "évaluation" and j["decision"] == "retenu"),
               key=lambda j: j["exactitude_stricte"])
atteinte = bool(meilleur["exactitude_stricte"] >= CIBLE_EXACTITUDE and meilleur["json_valide"] >= CIBLE_JSON)
print(f"Meilleure version retenue : {meilleur['version']} ({meilleur['exactitude_stricte']:.0%}) · cible atteinte : {atteinte}")

try:
    import ipywidgets as w
    w.interact(recommander,
               cible_atteinte=w.Checkbox(value=atteinte, description="Cible atteinte ?"),
               echecs_factuels=w.Checkbox(value=False, description="Échecs dus à des faits manquants ?"),
               constance_a_grande_echelle=w.Checkbox(value=True, description="Constance à grande échelle requise ?"),
               donnees_confidentielles=w.Checkbox(value=True, description="Données confidentielles ?"))
except Exception as e:   # ipywidgets absent ou incompatible : version statique
    print(f"Widgets indisponibles ({type(e).__name__}) : affichage statique.")
    recommander(atteinte, echecs_factuels=False, constance_a_grande_echelle=True, donnees_confidentielles=True)

<div style="background:linear-gradient(135deg,#00553A,#00A86A);color:#FFFFFF;padding:16px 22px;border-radius:10px;border-bottom:6px solid #F5C242">
<span style="color:#F5C242;font-size:30px;font-weight:800">07</span>&nbsp;&nbsp;<span style="font-size:22px;font-weight:700">🧪 À vous de jouer (≈ 15 minutes, en binômes)</span><br>
<i style="color:#E6F6EE">Optimiser le prompt de codification sur le jeu figé</i>
</div>

| Étape | Durée | Action |
|---|---|---|
| 1 | 2 min | Relire les erreurs restantes de la meilleure version (cellule ci-dessous) |
| 2 | 3 min | Nommer la cause des principales erreurs : ambiguïté, contexte ou format ? |
| 3 | 8 min | Modifier **une seule chose** dans `SYSTEME_V6`, exécuter, consigner ; recommencer |
| 4 | 2 min | Vérifier sur le jeu réservé, puis noter la meilleure version dans la feuille partagée |

<div style="background:#FBF1D9;border-left:5px solid #D49A00;padding:12px 16px;border-radius:6px">
<b>Règles du jeu :</b> ne jamais copier un cas du jeu d’évaluation dans le prompt ; un changement par version ; toute version est consignée, même si elle fait baisser le score.
</div>

In [ ]:
# 🔍 Étape 1 : erreurs restantes de la v4
afficher_erreurs(df_v4, 12)

In [ ]:
# ✏️ Étape 3 : votre version. Modifiez UNE chose, puis exécutez la cellule.
REGLES_V6 = REGLES + """
3. (À COMPLÉTER : votre nouvelle règle, ou supprimez cette ligne)"""

SYSTEME_V6 = SYSTEME_V1 + "\n" + REGLES_V6 + "\n" + TEXTE_EXEMPLES + "\n" + FORMAT_JSON

def messages_v6(texte):
    return [{"role": "system", "content": SYSTEME_V6},
            {"role": "user", "content": f"Description : {texte}"}]

VOTRE_CHANGEMENT = "décrivez ici votre changement en une phrase"
df_v6, res_v6 = evaluer("v6", messages_v6, json_mode=True, changement=VOTRE_CHANGEMENT)
afficher_erreurs(df_v6, 8)

<details>
<summary><b>💡 Indice (cliquez pour afficher)</b></summary>

Regardez le cas **L06** (*Shop assistant in a supermarket*) : le modèle le confond avec la vente sur les marchés. Une règle d’arbitrage explicite peut aider, par exemple :

`3. Les vendeurs et assistants de vente en magasin ou supermarché relèvent de 5223, pas de 5211 (réservé aux marchés et éventaires).`

Vérifiez ensuite sur le **jeu réservé** (cas H07) que le gain est réel.
</details>

In [ ]:
# ✅ Étape 4 : contrôle sur le jeu réservé et décision
_, res_v6_h = evaluer("v6", messages_v6, jeu=JEU_RESERVE, json_mode=True, changement="contrôle")
gain_eval = res_v6["exactitude_stricte"] - res_v4["exactitude_stricte"]
gain_reserve = res_v6_h["exactitude_stricte"] - res_v4_h["exactitude_stricte"]
print(f"Gain v6 vs v4 · jeu d'évaluation : {gain_eval:+.0%} · jeu réservé : {gain_reserve:+.0%}")
if gain_eval > 0 and gain_reserve >= 0:
    decider("v6", "retenu"); encadre("Gain confirmé sur les deux jeux : la v6 est <b>retenue</b>.", "note")
elif gain_eval > 0:
    decider("v6", "à vérifier"); encadre("Gain sur le jeu d'évaluation seulement : risque de sur-ajustement.", "attention")
else:
    decider("v6", "rejeté"); encadre("Pas de gain : la v6 est <b>rejetée</b>, mais elle reste consignée au journal.", "retenir")
afficher_journal()

<div style="background:#00704A;color:#FFFFFF;padding:14px 22px;border-radius:10px;border-left:10px solid #F5C242">
<span style="color:#F5C242;font-size:30px;font-weight:800">08</span>&nbsp;&nbsp;<span style="font-size:22px;font-weight:700">Synthèse, liste de contrôle et export</span><br>
<i style="color:#E6F6EE">Un bon prompt obtient une réponse. Un excellent prompt obtient une réponse défendable.</i>
</div>

### 8.1 Bilan des appels de cette session

In [ ]:
bilan = LLM.bilan()
display(bilan.style.format({"cout_usd": "{:.4f} $", "latence_moy_s": "{:.2f} s", "jetons_entree": "{:,.0f}", "jetons_sortie": "{:,.0f}"}))
total = bilan[["appels", "depuis_cache", "jetons_entree", "jetons_sortie", "cout_usd"]].sum()
print(f"Total : {total['appels']:.0f} appels dont {total['depuis_cache']:.0f} servis par le cache · "
      f"{total['jetons_entree']:,.0f} jetons d'entrée · {total['jetons_sortie']:,.0f} de sortie · "
      f"coût estimé {total['cout_usd']:.4f} $ (prix {'illustratifs' if MODE_SIMULATION else 'de la configuration'})")

### 8.2 La liste de contrôle du prompt excellent

La cellule suivante vérifie automatiquement ce qui peut l’être dans ce notebook.

In [ ]:
evals = [j for j in JOURNAL if j["jeu"] == "évaluation"]
retenus = [j for j in evals if j["decision"] == "retenu"]
meilleur = max(retenus, key=lambda j: j["exactitude_stricte"])
controles = [
    ("Évaluation", "Critères de succès et score cible écrits", "CIBLE_EXACTITUDE" in globals()),
    ("Évaluation", "Jeu figé et versionné (empreinte vérifiée)", empreinte(JEU_EVAL) == EMPREINTE_EVAL),
    ("Qualité", "Métrique adaptée à la tâche (stricte + par classe)", True),
    ("Qualité", "Journal : une version par changement, décisions notées", all(j["decision"] for j in evals)),
    ("Variance", "Température fixée à 0 pour la production", CONFIGURATION["temperature"] == 0),
    ("Variance", "Accord entre exécutions mesuré", "coh_t0" in globals()),
    ("Coût", "Coût pour 1 000 documents estimé", "scenarios_cout" in globals()),
    ("Coût", "Contenu stable en tête (système), cas à la fin", messages_v4("x")[-1]["content"].endswith("x")),
    ("Gouvernance", "Aucune fuite du jeu d'évaluation dans le prompt retenu",
     not any(f'"{t}"' in globals().get("SYSTEME_" + meilleur["version"].upper(), "") for t in JEU_EVAL["texte"])),
    ("Gouvernance", "Cible atteinte par la meilleure version retenue",
     bool(meilleur["exactitude_stricte"] >= CIBLE_EXACTITUDE and meilleur["json_valide"] >= CIBLE_JSON)),
]
lignes = "".join(
    f'<tr><td style="padding:4px 10px;color:{VERT_FONCE};font-weight:700">{g}</td><td style="padding:4px 10px">{t}</td>'
    f'<td style="padding:4px 10px;font-size:18px">{"✅" if ok else "⬜"}</td></tr>' for g, t, ok in controles)
display(HTML(f'<table style="border-collapse:collapse">{lignes}</table>'))
print(f"{sum(ok for *_, ok in controles)} / {len(controles)} contrôles validés · meilleure version retenue : {meilleur['version']}")

### 8.3 Exporter pour le banc d’essai de 14 h 45

Les fichiers suivants sont écrits dans le dossier `resultats_optimisation/` :
- `journal_iterations.csv` : l’historique complet ;
- `meilleur_prompt.json` : le prompt retenu et sa configuration, à réutiliser pour comparer les moteurs ;
- `jeu_evaluation.csv` et `jeu_reserve.csv` : les jeux figés, avec leur empreinte.

In [ ]:
from pathlib import Path
dossier = Path("resultats_optimisation"); dossier.mkdir(exist_ok=True)
pd.DataFrame(JOURNAL).to_csv(dossier / "journal_iterations.csv", index=False, encoding="utf-8-sig")
JEU_EVAL.to_csv(dossier / "jeu_evaluation.csv", index=False, encoding="utf-8-sig")
JEU_RESERVE.to_csv(dossier / "jeu_reserve.csv", index=False, encoding="utf-8-sig")
prompt_retenu = globals().get("SYSTEME_" + meilleur["version"].upper(), SYSTEME_V4)
(dossier / "meilleur_prompt.json").write_text(json.dumps(dict(
    version=meilleur["version"], systeme=prompt_retenu, gabarit_utilisateur="Description : {texte}",
    configuration=dict(CONFIGURATION, prompt_version=meilleur["version"]),
    scores=dict(exactitude_stricte=meilleur["exactitude_stricte"], json_valide=meilleur["json_valide"]),
    empreinte_jeu_eval=EMPREINTE_EVAL), ensure_ascii=False, indent=2), encoding="utf-8")
for f in sorted(dossier.iterdir()):
    print(f"📄 {f}  ({f.stat().st_size:,} octets)")
if EN_COLAB:
    print("Colab : ouvrez le panneau 📁 à gauche pour télécharger les fichiers.")

### 8.4 Messages clés

<div style="display:flex;flex-wrap:wrap;gap:10px">
<div style="flex:1;min-width:220px;background:#F4F7F5;border:1px solid #D5DED9;border-radius:10px;padding:14px">
<div style="font-size:30px;font-weight:800;color:#00704A">01</div><b>Mesurer avant de changer</b><br>Un jeu d’évaluation figé transforme les opinions en chiffres. Un changement par version, tout est consigné.</div>
<div style="flex:1;min-width:220px;background:#F4F7F5;border:1px solid #D5DED9;border-radius:10px;padding:14px">
<div style="font-size:30px;font-weight:800;color:#D49A00">02</div><b>Construire la reproductibilité</b><br>Température basse, modèle figé, format contraint, test de cohérence, vérification avant publication.</div>
<div style="flex:1;min-width:220px;background:#F4F7F5;border:1px solid #D5DED9;border-radius:10px;padding:14px">
<div style="font-size:30px;font-weight:800;color:#C4621D">03</div><b>Concevoir pour le coût</b><br>Structure du prompt, cache et lots peuvent diviser la facture par dix ; re-tester la qualité à chaque optimisation.</div>
</div>

<div style="background:#E8F5EF;border-left:5px solid #00A86A;padding:12px 16px;border-radius:6px;margin-top:12px">
<b>➡️ Ensuite, à 14 h 45 : « Choisir son moteur ».</b> Vous exécuterez <code>meilleur_prompt.json</code> chez deux fournisseurs, dont Groq, pour comparer latence, coût et qualité sur le même jeu d’évaluation.<br>
<b>📤 Partager :</b> déposez votre jeu d’évaluation et votre journal dans l’organisation GitHub de l’atelier, pour le manuel de référence du STG17 (activité 4.2.1).<br>
<b>🎯 Engagement :</b> avant la fin de la semaine, choisissez un prompt utilisé dans votre institut, construisez-lui un jeu de 30 cas et notez son score de référence.
</div>

### Références

- Assemblée générale des Nations Unies (2014). *Résolution 68/261, Principes fondamentaux de la statistique officielle.*
- Organisation internationale du Travail (2012). *Classification internationale type des professions, CITP-08.*
- Zheng, L. et al. (2023). *Judging LLM-as-a-Judge with MT-Bench and Chatbot Arena.* NeurIPS Datasets and Benchmarks.
- Wang, X. et al. (2022). *Self-Consistency Improves Chain of Thought Reasoning in Language Models.* arXiv:2203.11171.
- OpenAI Help Center. *What are tokens and how to count them?*
- Petrov, A. et al. (2023). *Language Model Tokenizers Introduce Unfairness Between Languages.* NeurIPS 2023.
- Liu, N. F. et al. (2023). *Lost in the Middle: How Language Models Use Long Contexts.* Transactions of the ACL (2024).
- Groq. *Supported Models* (console.groq.com/docs/models), consulté en septembre 2026.

<small><i>Données entièrement fictives. Libellés CITP-08 abrégés. En mode simulation, les scores proviennent d’un simulateur pédagogique et ne mesurent aucun modèle réel. Les prix sont illustratifs.</i></small>